# Unit 7. Generalised Linear Models I: Categorical Responses

精算統計模型 (SOA Exam SRM/PA). The text of [unit07.pdf](https://github.com/chang-ye-tu/asm/blob/master/note/unit07.pdf) with every R chunk as a code cell.

<a href="https://colab.research.google.com/github/chang-ye-tu/asm/blob/master/colab/unit07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The notebook runs on the R runtime (on Colab: Runtime › Change runtime type › R). Run the setup cell first: it installs the packages and downloads the data this unit uses.

In [ ]:
# Setup: run this cell first. It installs the packages this unit uses and downloads
# its data files into data/, the paths the code below reads.
local({
  if (is.null(getOption("repos")) || identical(unname(getOption("repos")["CRAN"]), "@CRAN@"))
    options(repos = c(CRAN = "https://cloud.r-project.org"))
  os <- tryCatch(readLines("/etc/os-release"), error = function(e) character())
  code <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
  if (length(code) == 1 && nzchar(code))  # Linux, e.g. Colab: prebuilt binary packages
    options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", code)),
            HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(), paste(getRversion(),
              R.version$platform, R.version$arch, R.version$os)))
})
pkgs <- c("boot", "class", "ISLR2", "MASS", "nnet")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new)) install.packages(new)
data_files <- c("data/frees/csv/AUTOBIsim.csv", "data/frees/csv/HKHorse.csv", "data/frees/csv/HealthExpend.csv", "data/frees/csv/TermLife.csv")
for (f in data_files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  url <- if (startsWith(f, "data/islr/"))
    paste0("https://www.statlearning.com/s/", basename(f)) else
    paste0("https://raw.githubusercontent.com/chang-ye-tu/asm/master/note/", f)
  download.file(url, f, quiet = TRUE, mode = "wb")
}
options(width = 62, digits = 4, scipen = 4, repr.plot.width = 6.4, repr.plot.height = 3.0)

In [ ]:
library(ISLR2)

## 1 Binary Responses

> **Reading.** FREES §11.1; ISLR §4.1–4.2 (PA reading list, not SRM). SRM learning outcomes 2(a), 2(c).

**Remark** (The classical assumptions; Unit 2). A1–A5 are Unit 2's classical assumptions on $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ with $\mathbf{X}$ fixed: linearity (A1), zero-mean errors (A2), $\operatorname{var}(\boldsymbol{\varepsilon})=\sigma^2\mathbf{I}_n$ with $\sigma^2>0$ (A3), $\operatorname{rank}(\mathbf{X})=k+1$, with $n>k+1$ for inference (A4), and normal errors (A5).

### 1.1 Binary variables

**Definition 7.1** (Binary response; FREES §11.1). A **binary** response takes the values $0$ and $1$. With $\pi_i=\operatorname{\mathsf{P}}(y_i=1)$ the response $y_i$ has the **Bernoulli** distribution with parameter $\pi_i$. Throughout, $y_1,\ldots,y_n$ are independent and $\mathbf{x}_i=(1,x_{i1},\ldots,x_{ik})'$ are fixed, the rows of an $n\times(k+1)$ matrix $\mathbf{X}$.

**Proposition 7.2** (Bernoulli moments; FREES §11.1). If $y$ is Bernoulli with parameter $\pi$, then $\operatorname{\mathsf{E}} y=\pi$ and $\operatorname{var} y=\pi(1-\pi)$, and $\pi$ determines the distribution.

**Proof.** $\operatorname{\mathsf{E}} y=0\cdot(1-\pi)+1\cdot\pi$, and $y^2=y$ gives $\operatorname{\mathsf{E}} y^2=\pi$, so $\operatorname{var} y=\pi-\pi^2$. The two probabilities are $\pi$ and $1-\pi$. ∎

**Example 7.1** (Hospitalisation by sex; FREES §11.1, Table 11.1). FREES's MEPS sample: $n=2000$ adults aged $18$ to $65$, with `POSEXP` $=1$ when inpatient expenditure is positive, that is, when the person was hospitalised. The $33$ persons of race `OTHER` are merged into `WHITE`, which reproduces FREES's tables, whose race variable has four levels:

In [ ]:
H <- read.csv("data/frees/csv/HealthExpend.csv")
H$POSEXP <- 1 * (H$EXPENDIP > 0)
table(H$RACE); H$RACE[H$RACE == "OTHER"] <- "WHITE"
H$RACE <- relevel(factor(H$RACE), ref = "WHITE"); H$REGION <- relevel(factor(H$REGION), ref = "WEST")
H$EDUC <- relevel(factor(H$EDUC), ref = "LHIGHSC"); H$PHSTAT <- relevel(factor(H$PHSTAT), ref = "EXCE")
H$INCOME <- relevel(factor(H$INCOME), ref = "POOR")
tab_sex <- table(POSEXP = H$POSEXP, GENDER = H$GENDER); tab_sex
round(100 * prop.table(tab_sex, 2), 1)

Of the $946$ men ($\texttt{GENDER}=0$), $44$ were hospitalised, $4.7\%$; of the $1054$ women, $113$, $10.7\%$.

### 1.2 The linear probability model

**Definition 7.3** (Linear probability model; FREES §11.1). The **linear probability model** is $y_i=\mathbf{x}_i'\boldsymbol{\beta}+\varepsilon_i$ with $\operatorname{\mathsf{E}}\varepsilon_i=0$, so that $\pi_i=\mathbf{x}_i'\boldsymbol{\beta}$, and it is fitted by least squares with $\operatorname{rank}(\mathbf{X})=k+1$.

**Proposition 7.4** (Three drawbacks; FREES §11.1). In the linear probability model with every $\pi_i$ in $(0,1)$:

(i) **fitted values can be poor:** if $\mathbf{x}'\boldsymbol{\beta}\in[0,1]$ for every $\mathbf{x}=(1,x_1,\ldots,x_k)'$ with $x_1,\ldots,x_k\in\mathbb{R}$, then $\beta_1=\cdots=\beta_k=0$; and a fitted value $\hat y_i=\mathbf{x}_i'\hat{\boldsymbol{\beta}}$ outside $[0,1]$ makes $\hat y_i(1-\hat y_i)$, the variance the fit implies, negative;

(ii) **heteroscedasticity:** $\operatorname{var}\varepsilon_i=\pi_i(1-\pi_i)$, so A3 holds only if every $\pi_i$ is $\pi$ or $1-\pi$ for one $\pi$;

(iii) **residual analysis is meaningless:** $\varepsilon_i$ takes the two values $1-\pi_i$ and $-\pi_i$, so A5 fails, and the residuals $e_i=y_i-\hat y_i$ lie on the two lines $e=1-\hat y$ and $e=-\hat y$.

**Proof.** (i) If $\beta_j\neq0$ for some $j\geqslant1$, the vector $\mathbf{x}$ with $x_j=t$ and the other slopes' coordinates $0$ gives $\mathbf{x}'\boldsymbol{\beta}=\beta_0+t\beta_j$, unbounded in $t$. And $t(1-t)<0$ exactly when $t<0$ or $t>1$.

(ii) $\varepsilon_i=y_i-\pi_i$ has the variance of $y_i$, which is $\pi_i(1-\pi_i)$ by Proposition 7.2; and $t(1-t)=s(1-s)$ iff $(t-s)(1-t-s)=0$.

(iii) $y_i$ is $1$ or $0$, so $\varepsilon_i$ is $1-\pi_i$ or $-\pi_i$ and $e_i$ is $1-\hat y_i$ or $-\hat y_i$. A normal variable has positive probability on every interval, and a variable with two values does not. ∎

**Remark** (FREES §11.1). Two remedies address (ii). Two-stage weighted least squares applies Unit 4's theorem **weighted least squares** with the weights $1/\{\hat y_i(1-\hat y_i)\}$ of a first least squares fit in place of the unknown $1/\{\pi_i(1-\pi_i)\}$, and needs every $\hat y_i$ in $(0,1)$. Least squares with robust standard errors needs nothing of the fitted values: as $|\varepsilon_i|\leqslant1$, the fourth moments are bounded and Unit 4's theorem **consistency of the robust standard errors** applies whenever its design conditions hold.

**Example 7.2** (The linear probability model on the MEPS data). Least squares of `POSEXP` on the explanatory variables of FREES's reduced model of §11.4 (sex, race, region, education, self-rated physical health, activity limitation, income and insurance), with HC0 standard errors:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
f_red <- POSEXP ~ GENDER + RACE + REGION + EDUC + PHSTAT + ANYLIMIT + INCOME + insure
lp <- lm(f_red, data = H); fv <- fitted(lp); el <- residuals(lp)
round(c(min_fitted = min(fv), max_fitted = max(fv)), 5); c(below_0 = sum(fv < 0), above_1 = sum(fv > 1))
Xl <- model.matrix(lp); Bl <- solve(crossprod(Xl))
se_hc0 <- sqrt(diag(Bl %*% crossprod(Xl * el) %*% Bl))
round(cbind(estimate = coef(lp), se_ols = coef(summary(lp))[, 2], se_hc0 = se_hc0)[c("GENDER", "PHSTATPOOR", "insure"), ], 4)
op <- par(mar = c(3.2, 3.8, 0.6, 0.6), mgp = c(2.1, 0.6, 0))
plot(fv, el, pch = 16, cex = 0.35, xlab = "fitted value", ylab = "residual"); abline(v = 0, lty = 3)
par(op)

*Figure 7.1.* Residuals against fitted values of the linear probability model for the MEPS data: the two lines $e=1-\hat y$ and $e=-\hat y$ of Proposition 7.4, with fitted values left of the dotted line below $0$.

The fitted values run from $-0.06903$ to $0.43648$, and $157$ are negative: the variances of Proposition 7.4(i) are negative there, and two-stage weighted least squares cannot be formed. The residuals lie on the two lines of (iii). The robust standard error of `PHSTATPOOR` is $0.0593$, against $0.0345$ from least squares.

**Proposition 7.5** (Coding a binary response; ISLR §4.2). In least squares with an intercept, replacing every $y_i$ by $1-y_i$ replaces every fitted value $\hat y_i$ by $1-\hat y_i$, so the classification “$1$ if $\hat y_i>1/2$” is the same under either coding, observations with $\hat y_i=1/2$ excepted; and $n^{-1}\sum_i\hat y_i=\bar y$, the proportion of ones.

**Proof.** $\hat{\mathbf{Y}}=\mathbf{H}\mathbf{Y}$ with $\mathbf{H}$ the hat matrix of Unit 2, the orthogonal projection onto $\mathcal{C}(\mathbf{X})$, which contains $\mathbf{1}$; so $\mathbf{H}\mathbf{1}=\mathbf{1}$ and $\mathbf{H}(\mathbf{1}-\mathbf{Y})=\mathbf{1}-\hat{\mathbf{Y}}$. And $\hat y_i>1/2$ iff $1-\hat y_i<1/2$. Finally $\mathbf{1}'\hat{\mathbf{Y}}=\mathbf{1}'\mathbf{H}\mathbf{Y}=(\mathbf{H}\mathbf{1})'\mathbf{Y}=\mathbf{1}'\mathbf{Y}$, $\mathbf{H}$ being symmetric. ∎

**Example 7.3** (Coding a three-level response; ISLR §4.2). A response with three unordered levels has no such invariance. `origin` in `Auto` is American, European or Japanese; least squares of a numerical code on `mpg`, `displacement`, `weight` and `year`, each fitted value classified to the level with the nearest code, under two codings:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
cl_or <- c("American", "European", "Japanese")[Auto$origin]
code_fit <- function(code) {
  f <- fitted(lm(code[cl_or] ~ mpg + displacement + weight + year, data = Auto))
  names(code)[sapply(f, function(v) which.min(abs(code - v)))] }
c1 <- code_fit(c(American = 1, European = 2, Japanese = 3))
c2 <- code_fit(c(Japanese = 1, American = 2, European = 3))
round(c(disagree = mean(c1 != c2), accuracy_first = mean(c1 == cl_or),
        accuracy_second = mean(c2 == cl_or)), 3)

The two codings classify $57.7\%$ of the $392$ cars differently, with accuracies $58.9\%$ and $62.2\%$. A numerical code imposes an order and equal spacing on the levels, and a different code is a different linear model.

**Example 7.4** (Credit card default: linear against logistic; ISLR §4.1–4.2, §4.3.1, Figure 4.2). ISLR's simulated `Default`: $10{,}000$ credit card holders, with `default` `Yes` or `No`, monthly `balance`, annual `income` and `student` status. Least squares and the logit on `balance`:

In [ ]:
Df <- Default; Df$y <- 1 * (Df$default == "Yes")
lpD <- lm(y ~ balance, data = Df); lgD <- glm(default ~ balance, family = binomial, data = Df)
round(c(default_rate = mean(Df$y), negative_fitted_lpm = mean(fitted(lpD) < 0),
        min_fitted_lpm = min(fitted(lpD)), mean_fitted_lpm = mean(fitted(lpD)),
        mean_fitted_logit = mean(fitted(lgD))), 4)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 2), mar = c(3.4, 3.8, 1.4, 0.6), mgp = c(2.1, 0.6, 0), cex = 0.8)
ob <- order(Df$balance)
for (m in list(lpD, lgD)) {
  plot(Df$balance, Df$y, pch = "|", cex = 0.5, col = "#B45309", xlab = "balance",
       ylab = "probability of default", ylim = c(-0.1, 1),
       main = if (inherits(m, "glm")) "logistic" else "least squares")
  lines(Df$balance[ob], fitted(m)[ob], col = "#1E3A8A", lwd = 2); abline(h = 0, lty = 3) }
par(op)

*Figure 7.2.* `Default`: fitted probabilities of default against `balance` by least squares (left) and by the logit (right), with the responses as ticks at $0$ and $1$ (ISLR Figure 4.2).

The default rate is $3.33\%$. Least squares gives negative fitted probabilities to $31.2\%$ of the card holders, down to $-0.0752$, the first drawback of Proposition 7.4; the logit's lie in $(0,1)$. The least squares fitted values average $0.0333$, the default rate, by Proposition 7.5.

## 2 Logistic and Probit Regression

> **Reading.** FREES §11.2; ISLR §4.3.1 (PA). SRM learning outcomes 2(a), 2(c).

### 2.1 Response functions

**Definition 7.6** (Binary regression; FREES §11.2.1). Let $\pi(\cdot)$ be a continuous, strictly increasing distribution function on $\mathbb{R}$. The **binary regression** with response function $\pi$ has $\pi_i=\operatorname{\mathsf{P}}(y_i=1)=\pi(\mathbf{x}_i'\boldsymbol{\beta})$, equivalently $\pi^{-1}(\pi_i)=\mathbf{x}_i'\boldsymbol{\beta}$, and $\pi^{-1}$ is the **link function**. The two cases used here are the **logit**, $\pi(z)=\Lambda(z)=1/(1+e^{-z})=e^z/(1+e^z)$, and the **probit**, $\pi(z)=\Phi(z)$, the standard normal distribution function. The choice $\pi(z)=z$ is the linear probability model, which is not a distribution function.

**Definition 7.7** (Odds and logit; FREES §11.2.4, ISLR (4.3)–(4.4)). For $p\in(0,1)$ the **odds** are $p/(1-p)$, and the **logit** is $\operatorname{logit}(p)=\ln\{p/(1-p)\}$, the log odds.

**Proposition 7.8** (The logistic function; FREES §11.2.4). $\Lambda$ is a continuous, strictly increasing distribution function, symmetric in the sense $\Lambda(-z)=1-\Lambda(z)$, with $\Lambda'=\Lambda(1-\Lambda)$ and $\operatorname{logit}\Lambda(z)=z$. So the logit model is $\operatorname{logit}\pi_i=\mathbf{x}_i'\boldsymbol{\beta}$.

**Proof.** $\Lambda'(z)=e^{-z}/(1+e^{-z})^2>0$, and $\Lambda(z)\to0$ and $1$ as $z\to-\infty$ and $+\infty$. Multiplying numerator and denominator of $\Lambda(-z)=1/(1+e^{z})$ by $e^{-z}$ gives $e^{-z}/(1+e^{-z})=1-\Lambda(z)$, and the product of $\Lambda(z)=1/(1+e^{-z})$ and $1-\Lambda(z)$ is $\Lambda'(z)$. Finally $\Lambda(z)/\{1-\Lambda(z)\}=e^z$. ∎

**Remark** (FREES §11.2.4). At odds against of $(1-p)/p$ to $1$, a unit stake on an event of probability $p$ wins $(1-p)/p$ with probability $p$ and is lost with probability $1-p$, an expected gain of $0$: odds are the payoffs of a fair bet.

### 2.2 Threshold and random utility

**Theorem 7.9** (Threshold interpretation; FREES §11.2.2). Let $y_i^*=\mathbf{x}_i'\boldsymbol{\beta}+\varepsilon_i^*$, where $\varepsilon_i^*$ has a continuous distribution function $G$ with $G(-a)=1-G(a)$ for all $a$, and let $y_i=\mathbb{1}\{y_i^*>0\}$. Then $\operatorname{\mathsf{P}}(y_i=1)=G(\mathbf{x}_i'\boldsymbol{\beta})$. If $\varepsilon_i^*=\sigma\varepsilon_i$ with $\sigma>0$ and $\varepsilon_i$ has such a distribution function $\pi$, then $\operatorname{\mathsf{P}}(y_i=1)=\pi(\mathbf{x}_i'\boldsymbol{\beta}/\sigma)$: the logit when $\varepsilon_i$ is logistic and the probit when it is standard normal, with $\boldsymbol{\beta}$ and $\sigma$ entering only through $\boldsymbol{\beta}/\sigma$.

**Proof.** $\operatorname{\mathsf{P}}(y_i=1)=\operatorname{\mathsf{P}}(\varepsilon_i^*>-\mathbf{x}_i'\boldsymbol{\beta})=1-G(-\mathbf{x}_i'\boldsymbol{\beta})=G(\mathbf{x}_i'\boldsymbol{\beta})$. With $\varepsilon_i^*=\sigma\varepsilon_i$, $G(a)=\operatorname{\mathsf{P}}(\varepsilon_i\leqslant a/\sigma)=\pi(a/\sigma)$, which inherits the symmetry. $\Lambda$ is symmetric by Proposition 7.8, and $\Phi$ because the normal density is even. ∎

**Remark**. Multiplying $\boldsymbol{\beta}$ and $\sigma$ by the same constant leaves every probability unchanged, so the data cannot identify the scale of the latent variable, and $\sigma=1$ is a normalisation.

**Proposition 7.10** (Random utility; FREES §11.2.3). Let $\varepsilon_{i1},\varepsilon_{i2}$ be independent with the extreme-value (Gumbel) distribution function $\operatorname{\mathsf{P}}(\varepsilon\leqslant a)=\exp(-e^{-a})$. Then $\varepsilon_{i2}-\varepsilon_{i1}$ has distribution function $\Lambda$. So if the utilities $U_{ij}=u_i(V_{ij}+\varepsilon_{ij})$, with $u_i$ strictly increasing, $V_{i1}=\mathbf{x}_i'\boldsymbol{\beta}$ and $V_{i2}=0$, and $y_i=1$ when $U_{i1}>U_{i2}$, then $\operatorname{\mathsf{P}}(y_i=1)=\Lambda(\mathbf{x}_i'\boldsymbol{\beta})$.

**Proof.** With $\gamma(a)=e^{-a}\exp(-e^{-a})$ the Gumbel density, independence and Fubini's theorem give

$$
\operatorname{\mathsf{P}}(\varepsilon_{i2}-\varepsilon_{i1}\leqslant t)
=\int_{-\infty}^{\infty}\exp\{-e^{-(t+a)}\}\,\gamma(a)\,\mathrm{d} a
=\int_0^\infty\exp\{-u(1+e^{-t})\}\,\mathrm{d} u=\frac1{1+e^{-t}} ,
$$

substituting $u=e^{-a}$. As $u_i$ is strictly increasing, $U_{i1}>U_{i2}$ iff $\varepsilon_{i2}-\varepsilon_{i1}<V_{i1}-V_{i2}=\mathbf{x}_i'\boldsymbol{\beta}$, an event of probability $\Lambda(\mathbf{x}_i'\boldsymbol{\beta})$, $\Lambda$ being continuous. ∎

### 2.3 Interpreting the coefficients

**Theorem 7.11** (Odds ratios and marginal effects; FREES §11.2.4, ISLR §4.3.1). In the logit model, with the other explanatory variables fixed:

(i) if $x_{ij}\in\{0,1\}$, then $e^{\beta_j}$ is the **odds ratio**, the odds of $y_i=1$ at $x_{ij}=1$ divided by the odds at $x_{ij}=0$;

(ii) $\beta_j=\partial\ln\operatorname{odds}(y_i=1)/\partial x_{ij}$, so a unit increase of $x_{ij}$ multiplies the odds by $e^{\beta_j}$;

(iii) $\partial\pi_i/\partial x_{ij}=\beta_j\pi_i(1-\pi_i)$, largest in size at $\pi_i=1/2$; in the probit model $\partial\pi_i/\partial x_{ij}=\beta_j\varphi(\mathbf{x}_i'\boldsymbol{\beta})$, with $\varphi$ the standard normal density.

**Proof.** By Proposition 7.8, $\ln\operatorname{odds}(y_i=1)=\mathbf{x}_i'\boldsymbol{\beta}$. For (i) subtract this identity at $x_{ij}=0$ from the identity at $x_{ij}=1$: the left sides differ by the logarithm of the odds ratio and the right sides by $\beta_j$. For (ii) differentiate it, and note that a unit increase adds $\beta_j$ to the log odds. For (iii) the chain rule gives $\beta_j\Lambda'(\mathbf{x}_i'\boldsymbol{\beta})=\beta_j\pi_i(1-\pi_i)$ by Proposition 7.8, and $t(1-t)$ is largest at $t=1/2$; for the probit, $\Phi'=\varphi$. ∎

**Remark** (FREES §11.2.4). $\beta_j$ is the proportional rate of change of the odds, what FREES calls the proportional change in the odds ratio; an elasticity in the economists' sense would be $\partial\ln\operatorname{odds}/\partial\ln x_{ij}=\beta_jx_{ij}$. An odds ratio is not a ratio of probabilities: $p_1/p_0$ and $\{p_1/(1-p_1)\}/\{p_0/(1-p_0)\}$ agree only when $p_1=p_0$ or in the limit of small probabilities.

**Example 7.5** (Odds of hospitalisation; FREES §11.2.4). The odds of Example 7.1, from the counts and from FREES's rounded percentages:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
p_w <- tab_sex["1", "1"] / sum(tab_sex[, "1"]); p_m <- tab_sex["1", "0"] / sum(tab_sex[, "0"])
round(c(odds_women = p_w / (1 - p_w), odds_men = p_m / (1 - p_m),
        odds_ratio = (p_w / (1 - p_w)) / (p_m / (1 - p_m)), probability_ratio = p_w / p_m), 4)
round(c(odds_women = 0.107 / 0.893, odds_men = 0.047 / 0.953, odds_ratio = (0.107 / 0.893) / (0.047 / 0.953),
        ratio_of_rounded_odds = 0.120 / 0.0493), 4)

From the counts, the odds are $0.1201$ for women and $0.0488$ for men, an odds ratio of $2.4617$, while the probabilities are in the ratio $2.3050$: women's odds are $2.4617$ times men's, their probability $2.3050$ times. FREES's odds $0.120$ and $0.0493$ come from the rounded percentages, $0.1198$ and $0.0493$ here, and its odds ratio $2.434$ is the ratio $2.4341$ of those rounded odds.

### 2.4 Logit against probit

**Proposition 7.12** (Logit against probit; FREES §11.2.1). $\Lambda$ and $\Phi$ are symmetric about $0$, with value $1/2$ there.

(i) $\Lambda(\beta_Lz)$ and $\Phi(\beta_Pz)$ have the same slope at $z=0$ iff $\beta_L=4\beta_P/\sqrt{2\pi}$, where $\pi=3.14159\ldots$ is the number.

(ii) The logistic distribution has mean $0$ and variance $\pi^2/3$; so if the latent error of Theorem 7.9 has standard deviation $s$, a logistic error gives the coefficients $\boldsymbol{\beta}_L=\pi\boldsymbol{\beta}/(\sqrt3\,s)$ and a normal one $\boldsymbol{\beta}_P=\boldsymbol{\beta}/s$, and $\boldsymbol{\beta}_L=\pi\boldsymbol{\beta}_P/\sqrt3$.

**Proof.** (i) $\Lambda'(0)=\Lambda(0)\{1-\Lambda(0)\}=1/4$ and $\Phi'(0)=1/\sqrt{2\pi}$, so the slopes are $\beta_L/4$ and $\beta_P/\sqrt{2\pi}$.

(ii) The density $\lambda(z)=\Lambda'(z)=e^{-z}/(1+e^{-z})^2$ is even by Proposition 7.8, and $\lambda(z)\leqslant e^{-|z|}$, so every moment exists and the mean is $0$. Integrating by parts, with $1-\Lambda$ an antiderivative of $-\lambda$ and $z^2\{1-\Lambda(z)\}\leqslant z^2e^{-z}\to0$, $\int_0^\infty z^2\lambda(z)\,\mathrm{d} z=2\int_0^\infty z\{1-\Lambda(z)\}\,\mathrm{d} z$. For $z>0$, $1-\Lambda(z)=e^{-z}/(1+e^{-z})=\sum_{m\geqslant1}(-1)^{m+1}e^{-mz}$, a geometric series whose partial sums $e^{-z}\{1-(-e^{-z})^N\}/(1+e^{-z})$ lie in $[0,e^{-z}]$; with the dominating function $ze^{-z}$, dominated convergence integrates it term by term:

$$
\operatorname{var}=2\int_0^\infty z^2\lambda(z)\,\mathrm{d} z=4\sum_{m\geqslant1}(-1)^{m+1}\int_0^\infty ze^{-mz}\,\mathrm{d} z
=4\sum_{m\geqslant1}\frac{(-1)^{m+1}}{m^2}=\frac{\pi^2}3
$$

by the prerequisites' lemma **sum of reciprocal squares**. By Theorem 7.9, a logistic latent error of standard deviation $s$ is $\sigma\varepsilon$ with $\varepsilon$ standard logistic and $\sigma=\sqrt3\,s/\pi$, which gives $\Lambda(\mathbf{x}'\boldsymbol{\beta}/\sigma)$, and a normal one is $sZ$, which gives $\Phi(\mathbf{x}'\boldsymbol{\beta}/s)$. ∎

**Example 7.6** (Similarity of the two functions; FREES §11.2.1, Figure 11.1). FREES's comparison of $\Lambda(\pi z/\sqrt3)$ with $\Phi(z)$, the two factors of Proposition 7.12, and the ratio $\operatorname{logit}(p)/\Phi^{-1}(p)$ over $0.1\leqslant p\leqslant0.9$:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
z <- seq(-4, 4, length = 2001); dl <- plogis(z * pi / sqrt(3)) - pnorm(z)
pp <- seq(0.1, 0.9, length = 801); pp <- pp[abs(pp - 0.5) > 1e-9]; rr <- qlogis(pp) / qnorm(pp)
round(c(max_abs_difference = max(abs(dl)), at_z = abs(z[which.max(abs(dl))]),
        slope_factor = 4 / sqrt(2 * pi), sd_factor = pi / sqrt(3), ratio_min = min(rr), ratio_max = max(rr)), 4)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.8, 0.6, 0.6), mgp = c(2.1, 0.6, 0))
plot(z, pnorm(z), type = "l", ylab = "distribution function"); lines(z, plogis(z * pi / sqrt(3)), lty = 2)
plot(z, dnorm(z), type = "l", ylab = "density", ylim = c(0, 0.47))
lines(z, pi / sqrt(3) * dlogis(z * pi / sqrt(3)), lty = 2)
par(op)

*Figure 7.3.* The standard normal (solid) and the logistic rescaled to unit variance (dashed): distribution functions (FREES Figure 11.1) and densities.

The rescaled logistic and the normal distribution functions differ by at most $0.0227$, at $|z|=0.684$, and on $0.1\leqslant p\leqslant0.9$ the ratio $\operatorname{logit}(p)/\Phi^{-1}(p)$ lies between $1.5958$, the slope factor of Proposition 7.12(i), and $1.7145$: over this range the two links are nearly proportional.

## 3 Inference for Logistic and Probit Regression

> **Reading.** FREES §11.3; ISLR §4.3.2–4.3.4 (PA). SRM learning outcomes 2(e), 2(f).

### 3.1 The likelihood and its maximiser

**Definition 7.13** (Log-likelihood; FREES §11.3.1, ISLR (4.5)). The log-likelihood of a binary regression is

$$
L(\boldsymbol{\beta})=\sum_{i=1}^n\bigl[y_i\ln\pi(\mathbf{x}_i'\boldsymbol{\beta})+(1-y_i)\ln\{1-\pi(\mathbf{x}_i'\boldsymbol{\beta})\}\bigr] ,
$$

the logarithm of the probability of the observed responses as a function of $\boldsymbol{\beta}$, and a maximiser $\mathbf{b}$ is a **maximum likelihood estimate**. When $\pi(-z)=1-\pi(z)$, as for the logit and the probit, the $i$th term is $\ln\pi(s_i\mathbf{x}_i'\boldsymbol{\beta})$ with $s_i=2y_i-1$.

**Theorem 7.14** (Score and information; FREES §11.3.1, §11.3.2). Let $\pi$ be twice continuously differentiable with $\pi'>0$, and write $\eta_i=\mathbf{x}_i'\boldsymbol{\beta}$ and $\pi_i=\pi(\eta_i)$. Then

$$
\nabla L(\boldsymbol{\beta})=\sum_{i=1}^n\mathbf{x}_i(y_i-\pi_i)\frac{\pi'(\eta_i)}{\pi_i(1-\pi_i)} ,
\qquad
\mathcal{I}(\boldsymbol{\beta})=\operatorname{var}\nabla L(\boldsymbol{\beta})=\sum_{i=1}^n\frac{\pi'(\eta_i)^2}{\pi_i(1-\pi_i)}\mathbf{x}_i\mathbf{x}_i' ,
$$

and $\mathcal{I}(\boldsymbol{\beta})=-\operatorname{\mathsf{E}}\nabla^2L(\boldsymbol{\beta})$, the **information matrix**. For the logit, $\nabla L(\boldsymbol{\beta})=\mathbf{X}'(\mathbf{Y}-\boldsymbol{\pi})$ and $-\nabla^2L(\boldsymbol{\beta})=\mathbf{X}'\mathbf{W}\mathbf{X}=\mathcal{I}(\boldsymbol{\beta})$ with $\mathbf{W}=\operatorname{diag}\{\pi_i(1-\pi_i)\}$, whatever the responses. For the probit, $-\nabla^2L(\boldsymbol{\beta})=\sum_iw_i\mathbf{x}_i\mathbf{x}_i'$ with the **observed weights**

$$
w_i=y_i\lambda_1(\eta_i)\{\lambda_1(\eta_i)+\eta_i\}+(1-y_i)\lambda_0(\eta_i)\{\lambda_0(\eta_i)-\eta_i\} ,
\qquad \lambda_1=\frac{\varphi}{\Phi},\quad \lambda_0=\frac{\varphi}{1-\Phi} .
$$

**Proof.** As $\partial\eta_i/\partial\boldsymbol{\beta}=\mathbf{x}_i$, the chain rule gives

$$
\nabla L=\sum_i\mathbf{x}_i\Bigl\{\frac{y_i}{\pi_i}-\frac{1-y_i}{1-\pi_i}\Bigr\}\pi'(\eta_i)
=\sum_i\mathbf{x}_i\frac{y_i(1-\pi_i)-(1-y_i)\pi_i}{\pi_i(1-\pi_i)}\pi'(\eta_i) ,
$$

with numerator $y_i-\pi_i$. The summands are independent with mean zero, and $\operatorname{var} y_i=\pi_i(1-\pi_i)$ by Proposition 7.2, so the variance of the score is the displayed sum; it equals $-\operatorname{\mathsf{E}}\nabla^2L$ by the prerequisites' lemma **score identities**, the responses taking two values. For the logit, $\pi'=\pi(1-\pi)$ by Proposition 7.8, which reduces the score to $\sum_i\mathbf{x}_i(y_i-\pi_i)$, and differentiating again, $\nabla^2L=-\sum_i\pi_i(1-\pi_i)\mathbf{x}_i\mathbf{x}_i'$, free of the responses. For the probit, with $\varphi'(\eta)=-\eta\varphi(\eta)$, $(\ln\Phi)''=(\varphi'\Phi-\varphi^2)/\Phi^2=-\lambda_1(\lambda_1+\eta)$ and $\{\ln(1-\Phi)\}''=-\{\varphi'(1-\Phi)+\varphi^2\}/(1-\Phi)^2=-\lambda_0(\lambda_0-\eta)$, so the second derivative of the $i$th term in $\eta_i$ is $-w_i$. ∎

**Theorem 7.15** (Concavity). For the logit and the probit, each term of $L$ is concave in $\boldsymbol{\beta}$, and $L$ is strictly concave when $\operatorname{rank}(\mathbf{X})=k+1$.

**Proof.** **Logit.** $\ln\Lambda(\eta)=-\ln(1+e^{-\eta})$ has second derivative $-\Lambda(\eta)\{1-\Lambda(\eta)\}<0$, and $\ln\{1-\Lambda(\eta)\}=\ln\Lambda(-\eta)$ has the same second derivative, the chain rule contributing $(-1)^2$.

**Probit.** By Unit 5's lemma **truncated normal moments**, $\lambda_0$ is differentiable with $\lambda_0'\in(0,1)$; as $\{\ln(1-\Phi)\}'=-\lambda_0$, the function $\ln(1-\Phi)$ has second derivative $-\lambda_0'<0$, and so does $\ln\Phi(\eta)=\ln\{1-\Phi(-\eta)\}$, at $-\eta$.

**Both.** So the $i$th term $\ell_i(\eta)$ has $\ell_i''<0$. Along a line, $t\mapsto\ell_i(\mathbf{x}_i'\boldsymbol{\beta}+t\mathbf{x}_i'\mathbf{d})$ has second derivative $\ell_i''(\cdot)(\mathbf{x}_i'\mathbf{d})^2\leqslant0$, and $t\mapsto L(\boldsymbol{\beta}+t\mathbf{d})$ has second derivative $-\sum_ic_i(\mathbf{x}_i'\mathbf{d})^2$ with $c_i=-\ell_i''(\mathbf{x}_i'\boldsymbol{\beta}+t\mathbf{x}_i'\mathbf{d})>0$, negative iff $\mathbf{X}\mathbf{d}\neq\mathbf{0}$, which holds for every $\mathbf{d}\neq\mathbf{0}$ iff $\operatorname{rank}(\mathbf{X})=k+1$. By the prerequisites' lemma **convexity from the derivative**, each term is concave and, when $\operatorname{rank}(\mathbf{X})=k+1$, $L$ is strictly concave. ∎

**Corollary 7.16** (The logit score equations; FREES §11.3.1). A logit maximum likelihood estimate $\mathbf{b}$, with fitted values $\hat y_i=\Lambda(\mathbf{x}_i'\mathbf{b})$, satisfies $\sum_ix_{ij}\hat y_i=\sum_ix_{ij}y_i$ for every $j$; the intercept gives $\sum_i\hat y_i=\sum_iy_i$. If $\mathbf{x}_i=(1,x_i)'$ with $x_i\in\{0,1\}$, and the $n_1$ observations with $x_i=0$ and the $n_2$ with $x_i=1$ have means $\bar y_1,\bar y_2\in(0,1)$, then the estimate is $b_0=\operatorname{logit}\bar y_1$, $b_1=\operatorname{logit}\bar y_2-\operatorname{logit}\bar y_1$, and $e^{b_1}$ is the sample odds ratio.

**Proof.** A maximiser of the differentiable $L$ has $\nabla L=\mathbf{X}'(\mathbf{Y}-\hat{\mathbf{Y}})=\mathbf{0}$ by Theorem 7.14, which read coordinate by coordinate is the first statement, and the intercept column is $\mathbf{1}$. In the two-group case the fitted values are $\Lambda(b_0)$ and $\Lambda(b_0+b_1)$ in the two groups, and the score equations for the intercept and for $x_i$ say $n_1\Lambda(b_0)+n_2\Lambda(b_0+b_1)=n_1\bar y_1+n_2\bar y_2$ and $n_2\Lambda(b_0+b_1)=n_2\bar y_2$, so $\Lambda(b_0)=\bar y_1$ and $\Lambda(b_0+b_1)=\bar y_2$, solved by the stated values by Proposition 7.8. These solve $\nabla L=\mathbf{0}$, and a stationary point of the concave $L$ (Theorem 7.15) is a maximiser by the prerequisites' proposition **first-order condition**; $\operatorname{rank}(\mathbf{X})=2$ makes $L$ strictly concave, so it is the only one, as two maximisers would give a larger value at their midpoint. Finally $e^{b_1}=\{\bar y_2/(1-\bar y_2)\}/\{\bar y_1/(1-\bar y_1)\}$. ∎

**Example 7.7** (Sex alone; FREES §11.3.1). The logit fit of `POSEXP` on `GENDER` for the data of Example 7.1:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
g1 <- glm(POSEXP ~ GENDER, family = binomial, data = H)
round(c(b0 = coef(g1)[[1]], logit_men = qlogis(p_m), b1 = coef(g1)[[2]], exp_b1 = exp(coef(g1)[[2]]),
        sample_odds_ratio = (p_w / (1 - p_w)) / (p_m / (1 - p_m))), 4)

$b_0=-3.0204$ is the logit of the men's proportion, and $e^{b_1}=2.4617$ is the sample odds ratio of Example 7.5, as Corollary 7.16 states.

**Theorem 7.17** (Existence and separation). Let $L$ be a logit or probit log-likelihood with $\operatorname{rank}(\mathbf{X})=k+1$, and $s_i=2y_i-1$.

(i) If some $\mathbf{c}\neq\mathbf{0}$ has $s_i\mathbf{x}_i'\mathbf{c}\geqslant0$ for every $i$, no maximum likelihood estimate exists. If all these inequalities are strict (the data are **separated**), $L(t\mathbf{c})$ increases to $\sup L=0$ as $t\to\infty$.

(ii) Otherwise $L$ has exactly one maximiser.

**Proof.** By Definition 7.13, $L(\boldsymbol{\beta})=\sum_i\ln\pi(s_i\mathbf{x}_i'\boldsymbol{\beta})$ with $\ln\pi$ strictly increasing and negative.

(i) For any $\boldsymbol{\beta}$, $L(\boldsymbol{\beta}+t\mathbf{c})=\sum_i\ln\pi(s_i\mathbf{x}_i'\boldsymbol{\beta}+ts_i\mathbf{x}_i'\mathbf{c})$ is nondecreasing in $t$, and strictly increasing because some $s_i\mathbf{x}_i'\mathbf{c}>0$: otherwise $\mathbf{X}\mathbf{c}=\mathbf{0}$, against $\operatorname{rank}(\mathbf{X})=k+1$. So no $\boldsymbol{\beta}$ maximises $L$. With strict inequalities each term of $L(t\mathbf{c})$ increases to $\ln1=0$, while $L<0$ everywhere.

(ii) Now every unit vector $\mathbf{c}$ has some $s_i\mathbf{x}_i'\mathbf{c}<0$, so $m(\mathbf{c})=\max_i(-s_i\mathbf{x}_i'\mathbf{c})>0$. The function $m$ is continuous on the unit sphere, which is compact by the prerequisites' theorem **Heine–Borel**, so it attains a minimum $m_0>0$ by the prerequisites' theorem **uniform continuity on compacts**. Keeping only the term at which the maximum is attained, and using that $-\ln\pi$ is decreasing and nonnegative,

$$
-L(t\mathbf{c})=\sum_i\{-\ln\pi(ts_i\mathbf{x}_i'\mathbf{c})\}\geqslant-\ln\pi(-tm(\mathbf{c}))\geqslant-\ln\pi(-tm_0)
\longrightarrow\infty
$$

as $t\to\infty$, uniformly in $\mathbf{c}$. So $-L$ is continuous and coercive and has a minimiser by the prerequisites' lemma **coercive minimisation**; it is strictly convex by Theorem 7.15, so the minimiser is unique by the same lemma. ∎

**Example 7.8** (A separated sample). Six observations with $y_i=\mathbb{1}\{x_i>3.5\}$ are separated by $\mathbf{c}=(-3.5,1)'$:

In [ ]:
xs <- 1:6; ys <- c(0, 0, 0, 1, 1, 1)
gs <- suppressWarnings(glm(ys ~ xs, family = binomial))
round(c(slope = coef(gs)[[2]], se = coef(summary(gs))[2, 2], iterations = gs$iter,
        max_fitted_y0 = max(fitted(gs)[ys == 0]), min_fitted_y1 = min(fitted(gs)[ys == 1])), 6)
Ls <- function(t) sum(plogis((2 * ys - 1) * t * (xs - 3.5), log.p = TRUE))
round(c(t1 = Ls(1), t5 = Ls(5)), 4); signif(c(t20 = Ls(20)), 3)

`glm` stops after $25$ iterations with slope $47.23$, standard error $115264.41$ and fitted values $0.00$ and $1.00$ to the printed digits: a symptom of Theorem 7.17(i), not an estimate. Along $\mathbf{c}$ the log-likelihood rises towards $0$: $-1.5088$, $-0.1589$ and $-0.0000908$ at $t=1$, $5$ and $20$.

### 3.2 Fisher scoring (beyond the syllabus)

> **Reading.** Beyond the syllabus: FREES §11.9.2, the chapter's technical supplement, which is outside the reading list.

**Theorem 7.18** (Fisher scoring is iteratively reweighted least squares). Let $\pi$ be as in Theorem 7.14, $\operatorname{rank}(\mathbf{X})=k+1$, and at a current value $\boldsymbol{\beta}^{(m)}$ let $\eta_i=\mathbf{x}_i'\boldsymbol{\beta}^{(m)}$, $\pi_i=\pi(\eta_i)$, $f_i=\pi'(\eta_i)$, $\omega_i=f_i^2/\{\pi_i(1-\pi_i)\}$ and $z_i=\eta_i+(y_i-\pi_i)/f_i$. The Fisher scoring step $\boldsymbol{\beta}^{(m)}+\mathcal{I}(\boldsymbol{\beta}^{(m)})^{-1}\nabla L(\boldsymbol{\beta}^{(m)})$ is the weighted least squares fit $(\mathbf{X}'\boldsymbol{\Omega}\mathbf{X})^{-1}\mathbf{X}'\boldsymbol{\Omega}\mathbf{z}$, $\boldsymbol{\Omega}=\operatorname{diag}(\omega_i)$. For the logit it is also the Newton–Raphson step $\boldsymbol{\beta}^{(m)}-\{\nabla^2L(\boldsymbol{\beta}^{(m)})\}^{-1}\nabla L(\boldsymbol{\beta}^{(m)})$.

**Proof.** By Theorem 7.14, $\mathcal{I}=\mathbf{X}'\boldsymbol{\Omega}\mathbf{X}$, invertible as $\operatorname{rank}(\mathbf{X})=k+1$ and every $\omega_i>0$, and $\nabla L=\sum_i\mathbf{x}_i\omega_i(y_i-\pi_i)/f_i=\mathbf{X}'\boldsymbol{\Omega}(\mathbf{z}-\mathbf{X}\boldsymbol{\beta}^{(m)})$. So $\boldsymbol{\beta}^{(m)}+(\mathbf{X}'\boldsymbol{\Omega}\mathbf{X})^{-1}\mathbf{X}'\boldsymbol{\Omega}(\mathbf{z}-\mathbf{X}\boldsymbol{\beta}^{(m)})=(\mathbf{X}'\boldsymbol{\Omega}\mathbf{X})^{-1}\mathbf{X}'\boldsymbol{\Omega}\mathbf{z}$. For the logit $-\nabla^2L=\mathcal{I}$ by the same theorem. ∎

### 3.3 Large-sample inference

**Theorem 7.19** (Large-sample distribution; FREES §11.3.2). Let the responses follow the logit or the probit model with parameter $\boldsymbol{\beta}_0$, and let the design satisfy

(D1) $|\mathbf{x}_i|\leqslant M$ for every $i$, with $M$ free of $n$;

(D2) $n^{-1}\mathcal{I}(\boldsymbol{\beta}_0)\to\mathbf{F}$, positive definite.

Then with probability tending to one the maximum likelihood estimate $\mathbf{b}$ exists and is unique, $\mathbf{b}\xrightarrow{\,p\,}\boldsymbol{\beta}_0$, $\sqrt n(\mathbf{b}-\boldsymbol{\beta}_0)\xrightarrow{\,d\,} N(\mathbf{0},\mathbf{F}^{-1})$, and both $n^{-1}\mathcal{I}(\mathbf{b})$ and $-n^{-1}\nabla^2L(\mathbf{b})$ converge in probability to $\mathbf{F}$.

**Proof.** The hypotheses of the prerequisites' theorem **concave maximum likelihood** are checked with $\Theta=\mathbb{R}^{k+1}$ and $\ell_i(\boldsymbol{\beta})=y_i\ln\pi(\mathbf{x}_i'\boldsymbol{\beta})+(1-y_i)\ln\{1-\pi(\mathbf{x}_i'\boldsymbol{\beta})\}$. Write $c(\eta;y)$ for minus the second derivative in $\eta$ of $y\ln\pi(\eta)+(1-y)\ln\{1-\pi(\eta)\}$: $\Lambda(\eta)\{1-\Lambda(\eta)\}$ for the logit and the observed weight of Theorem 7.14 for the probit. By the prerequisites' theorem **Cauchy–Schwarz**, $|\mathbf{x}_i'\boldsymbol{\beta}|\leqslant M(|\boldsymbol{\beta}_0|+1)$ whenever $|\boldsymbol{\beta}-\boldsymbol{\beta}_0|\leqslant1$; call this interval of $\eta$ values $J$.

**Concavity and smoothness.** Each $\ell_i$ is concave by Theorem 7.15 and twice continuously differentiable.

**The score.** By Theorem 7.14, $\nabla\ell_i(\boldsymbol{\beta}_0)=\mathbf{x}_ir(\eta_i)(y_i-\pi_i)$ with $r=\pi'/\{\pi(1-\pi)\}$, which has mean zero and variance $\pi'(\eta_i)^2\mathbf{x}_i\mathbf{x}_i'/\{\pi_i(1-\pi_i)\}$, so the average variance is $n^{-1}\mathcal{I}(\boldsymbol{\beta}_0)\to\mathbf{F}$ by (D2). The continuous $r$ is bounded on the compact $J$, by $R_0$ say, by the prerequisites' theorem **uniform continuity on compacts**, so $|\nabla\ell_i(\boldsymbol{\beta}_0)|\leqslant MR_0$, and the prerequisites' corollary **bounded summands with unequal variances** gives $n^{-1/2}\nabla L(\boldsymbol{\beta}_0)\xrightarrow{\,d\,} N(\mathbf{0},\mathbf{F})$.

**The curvature.** $-\nabla^2L(\boldsymbol{\beta}_0)=\sum_ic(\eta_i;y_i)\mathbf{x}_i\mathbf{x}_i'$. For the logit it equals $\mathcal{I}(\boldsymbol{\beta}_0)$. For the probit, $\operatorname{\mathsf{E}} c(\eta_i;y_i)=\pi_i\lambda_1(\lambda_1+\eta_i)+(1-\pi_i)\lambda_0(\lambda_0-\eta_i) =\varphi^2/\Phi+\varphi^2/(1-\Phi)$ at $\eta_i$, which is the weight of $\mathcal{I}(\boldsymbol{\beta}_0)$; each entry of $n^{-1}\{-\nabla^2L(\boldsymbol{\beta}_0)-\mathcal{I}(\boldsymbol{\beta}_0)\}$ is an average of independent mean-zero terms bounded by a constant times $M^2$, so its variance is of order $1/n$ and it tends to $0$ in probability by the prerequisites' theorem **Markov, Chebyshev**. In both cases $-n^{-1}\nabla^2L(\boldsymbol{\beta}_0)\xrightarrow{\,p\,}\mathbf{F}$. The derivative of $c(\cdot;y)$ is continuous, hence bounded on $J$ by some $K_0$ for $y=0,1$ by the same theorem, so the prerequisites' mean value theorem gives $|c(\mathbf{x}_i'\boldsymbol{\beta};y_i)-c(\mathbf{x}_i'\boldsymbol{\beta}_0;y_i)|\leqslant K_0M|\boldsymbol{\beta}-\boldsymbol{\beta}_0|$ and, with $\|\mathbf{x}_i\mathbf{x}_i'\|=|\mathbf{x}_i|^2\leqslant M^2$,

$$
\|\nabla^2L(\boldsymbol{\beta})-\nabla^2L(\boldsymbol{\beta}_0)\|\leqslant nK_0M^3|\boldsymbol{\beta}-\boldsymbol{\beta}_0|,\qquad |\boldsymbol{\beta}-\boldsymbol{\beta}_0|\leqslant1 .
$$

The same argument applies to $\mathcal{I}(\boldsymbol{\beta})$, whose weights $\pi'^2/\{\pi(1-\pi)\}$ have a continuous derivative.

**Conclusion.** The limiting curvature equals the limiting variance of the score, so the theorem gives existence with probability tending to one, consistency, the normal limit with variance $\mathbf{F}^{-1}$, and $-n^{-1}\nabla^2L(\mathbf{b})\xrightarrow{\,p\,}\mathbf{F}$; then $\|n^{-1}\mathcal{I}(\mathbf{b})-n^{-1}\mathcal{I}(\boldsymbol{\beta}_0)\|\leqslant K_0'M^3|\mathbf{b}-\boldsymbol{\beta}_0|\xrightarrow{\,p\,}0$ for the constant $K_0'$ of $\mathcal{I}$. For uniqueness, (D2) makes $n^{-1}\mathbf{d}'\mathcal{I}(\boldsymbol{\beta}_0)\mathbf{d}>0$ for every $\mathbf{d}\neq\mathbf{0}$ once $n$ is large, which forces $\operatorname{rank}(\mathbf{X})=k+1$; $L$ is then strictly concave by Theorem 7.15, and two maximisers would give a larger value at their midpoint. ∎

**Remark** (FREES §11.3.2). So $\mathbf{b}$ is approximately $N(\boldsymbol{\beta}_0,\mathcal{I}(\mathbf{b})^{-1})$, and $\operatorname{se}(b_j)$ is the square root of the $(j+1)$st diagonal element of $\mathcal{I}(\mathbf{b})^{-1}$; FREES's text says “of this matrix”, and its errata correct this to the inverse. The ratio $z=b_j/\operatorname{se}(b_j)$ is referred to $N(0,1)$ when $\beta_j=0$; R's `glm` computes $\operatorname{se}$ from the expected information $\mathcal{I}(\mathbf{b})$, which for the logit is also the observed $-\nabla^2L(\mathbf{b})$ but not for the probit, where either may be used.

**Definition 7.20** (Wald and likelihood ratio tests; FREES §11.3.2). For $H_0\colon\mathbf{R}\boldsymbol{\beta}=\mathbf{r}$ with $\mathbf{R}$ a $q\times(k+1)$ matrix of rank $q$, let $\mathbf{b}_R$ maximise $L$ subject to $\mathbf{R}\boldsymbol{\beta}=\mathbf{r}$. The **Wald** and **likelihood ratio** statistics are

$$
W=(\mathbf{R}\mathbf{b}-\mathbf{r})'\bigl\{\mathbf{R}\mathcal{I}(\mathbf{b})^{-1}\mathbf{R}'\bigr\}^{-1}(\mathbf{R}\mathbf{b}-\mathbf{r}) ,
\qquad
\text{LRT}=2\{L(\mathbf{b})-L(\mathbf{b}_R)\} .
$$

For the overall hypothesis $\beta_1=\cdots=\beta_k=0$, $L(\mathbf{b}_R)$ is the maximised log-likelihood $L_0$ of the intercept-only model and $q=k$.

**Corollary 7.21** (Chi-square limits; FREES §11.3.2). Under Theorem 7.19 and $H_0$, $W$ and $\text{LRT}$ converge in distribution to $\chi^2_q$. For $H_0\colon\beta_j=0$, $W=z^2$.

**Proof.** By Theorem 7.19 and the prerequisites' theorems **continuous mapping** and **closure under affine maps**, $\sqrt n(\mathbf{R}\mathbf{b}-\mathbf{r})=\mathbf{R}\sqrt n(\mathbf{b}-\boldsymbol{\beta}_0)\xrightarrow{\,d\,} N(\mathbf{0},\mathbf{R}\mathbf{F}^{-1}\mathbf{R}')$, with $\mathbf{R}\mathbf{F}^{-1}\mathbf{R}'$ positive definite as $\mathbf{R}$ has rank $q$; and $\mathbf{R}\{n^{-1}\mathcal{I}(\mathbf{b})\}^{-1}\mathbf{R}'\xrightarrow{\,p\,}\mathbf{R}\mathbf{F}^{-1}\mathbf{R}'$ by the prerequisites' lemma **inverses of convergent positive definite matrices** and theorem **continuous mapping**. As $W=\{\sqrt n(\mathbf{R}\mathbf{b}-\mathbf{r})\}'[\mathbf{R}\{n^{-1}\mathcal{I}(\mathbf{b})\}^{-1}\mathbf{R}']^{-1} \{\sqrt n(\mathbf{R}\mathbf{b}-\mathbf{r})\}$, the prerequisites' theorem **quadratic forms of asymptotically normal vectors** gives $W\xrightarrow{\,d\,}\chi^2_q$. For the likelihood ratio, complete $\mathbf{R}$ to an invertible matrix $\mathbf{T}$ whose last $q$ rows are $\mathbf{R}$ and write $\boldsymbol{\gamma}=\mathbf{T}\boldsymbol{\beta}$: the model is a binary regression in $\boldsymbol{\gamma}$ with rows $(\mathbf{T}')^{-1}\mathbf{x}_i$, which satisfy (D1) and (D2), and $H_0$ fixes the last $q$ coordinates of $\boldsymbol{\gamma}$. The restricted problem is a binary regression in the first $k+1-q$ coordinates with a known offset, and the proof of Theorem 7.19 applies to it unchanged. The prerequisites' theorem **likelihood ratio** then gives $\text{LRT}\xrightarrow{\,d\,}\chi^2_q$. For $q=1$ and $\mathbf{R}$ the $j$th unit row, $W=b_j^2/[\mathcal{I}(\mathbf{b})^{-1}]_{jj}=z^2$. ∎

**Proposition 7.22** (The likelihood ratio in the normal linear model). Under A1–A5 and the true hypothesis that $q$ given slopes are zero, with $k$ fixed, the likelihood ratio statistic of the submodel without them against the full model, twice the difference of the maximised normal log-likelihoods, converges in distribution to $\chi^2_q$ as $n\to\infty$.

**Proof.** With $m=n-k-1$, Unit 5's proposition **likelihood ratio and $F$** gives $\text{LR}=n\ln(1+qF/m)$ with $F=\{(\mathrm{RSS}_R-\mathrm{RSS}_U)/q\}/(\mathrm{RSS}_U/m)$, which by Unit 3's corollary **extra sum of squares** is the statistic of Unit 3's theorem **the general linear hypothesis**: defined with probability one, $\mathrm{RSS}_U>0$, and under $H_0$ of law $F_{q,m}$, the law of $(A/q)/(B/m)$ with $A\sim\chi^2_q$ and $B\sim\chi^2_m$ independent. By the prerequisites' proposition **moments of $\chi^2$**, $B/m$ has mean $1$ and variance $2/m$, so $B/m\xrightarrow{\,p\,}1$ by the prerequisites' theorem **Markov, Chebyshev** and $qF\xrightarrow{\,d\,}\chi^2_q$ by the prerequisites' theorem **Slutsky**; hence $u_n=qF/m\xrightarrow{\,d\,}0$, so $u_n\xrightarrow{\,p\,}0$ by the prerequisites' proposition **convergence to a constant**. With $h(u)=\ln(1+u)/u$ for $u>-1$, $u\neq0$, and $h(0)=1$, continuous, $\text{LR}=(n/m)\,qF\,h(u_n)$, and the same theorem with $n/m\to1$ and $h(u_n)\xrightarrow{\,p\,}1$, by the prerequisites' theorem **continuous mapping**, gives $\text{LR}\xrightarrow{\,d\,}\chi^2_q$. ∎

**Definition 7.23** (Information criterion; FREES §11.4, Table 11.5). For a model fitted by maximum likelihood with $d$ estimated parameters, $\mathrm{AIC}=-2L(\mathbf{b})+2d$. For ordinary linear models the SRM syllabus uses instead the form of Unit 5's definition **$C_p$, AIC, BIC and adjusted $R^2$**.

**Definition 7.24** (Measures of fit; FREES §11.3.2). With $L_0$ the maximised intercept-only log-likelihood, the **max-scaled** $R^2$ is $R^2_{ms}=R^2/R^2_{\max}$, where $R^2=1-\exp\{2(L_0-L(\mathbf{b}))/n\}$ and $R^2_{\max}=1-\exp(2L_0/n)$; FREES prints $R^2$ without the factor $2$ in the exponent, and its errata restore it. The **pseudo**-$R^2$ is $\{L(\mathbf{b})-L_0\}/(L_{\max}-L_0)$, with $L_{\max}$ the supremum of the log-likelihood over all fitted probabilities in $(0,1)^n$.

**Proposition 7.25** (Range of the measures; FREES §11.3.2). For binary responses, not all equal, a model with an intercept and its maximum likelihood estimate $\mathbf{b}$, $L_{\max}=0$ and $L_0\leqslant L(\mathbf{b})<0$, so the pseudo-$R^2$ and $R^2_{ms}$ lie in $[0,1]$, and $\text{LRT}=2\{L(\mathbf{b})-L_0\}=-2L_0\times\text{pseudo-}R^2$.

**Proof.** Each term $y_i\ln p_i+(1-y_i)\ln(1-p_i)$ is negative for $p_i\in(0,1)$ and tends to $0$ as $p_i\to y_i$, so $L_{\max}=0$ and $L(\mathbf{b})<0$. The intercept-only model is the model with $\beta_1=\cdots=\beta_k=0$, so $L(\mathbf{b})\geqslant L_0$. Hence $\{L(\mathbf{b})-L_0\}/(0-L_0)\in[0,1]$, and $2(L_0-L(\mathbf{b}))/n\in[2L_0/n,0]$ gives $0\leqslant R^2\leqslant R^2_{\max}$, with $R^2_{\max}>0$ as $L_0<0$. The last identity is the definition of the pseudo-$R^2$ with $L_{\max}=0$. ∎

**Example 7.9** (Job security; FREES §11.3.2, Table 11.3). Valletta's probit of dismissal on $n=24{,}168$ PSID observations reports, among other coefficients, tenure $-0.084$, tenure $\times$ time trend $0.003$, change in logarithmic sector employment $0.094$, and tenure $\times$ that change $-0.020$, with $-2L(\mathbf{b})=7027.8$ and pseudo-$R^2$ $0.097$. The tenure effect on the index in year $t$ of the trend, the sector employment effect at the mean tenure $7.8$, and the likelihood ratio statistic of Proposition 7.25:

In [ ]:
L_job <- -7027.8 / 2; L0_job <- L_job / (1 - 0.097)
round(c(tenure_effect_t0 = -0.084, tenure_effect_t16 = -0.084 + 0.003 * 16,
        sector_effect_tenure0 = 0.094, sector_effect_tenure7.8 = 0.094 - 0.020 * 7.8,
        L0 = L0_job, LRT = -2 * L0_job * 0.097), 3)

The tenure effect on the dismissal index, $-0.084$ at the start of the trend, is $-0.036$ in 1992 ($t=16$): dismissal falls with tenure less steeply than before. The sector employment effect, $0.094$ at tenure $0$, is $-0.062$ at the mean tenure $7.8$: for experienced men a fall in sector employment raises the dismissal index, FREES's reading. With $L_0=-3891.362$ the overall likelihood ratio statistic is $754.924$.

**Example 7.10** (Credit card default; ISLR §4.3.2–4.3.3, Tables 4.1–4.2). The logits of `default` on `balance` and on `student` alone, and ISLR's predictions, from the fits and from ISLR's rounded coefficients:

In [ ]:
gD2 <- glm(default ~ student, family = binomial, data = Df)
round(coef(summary(lgD)), 5); round(coef(summary(gD2)), 4)
round(c(balance_1000 = predict(lgD, data.frame(balance = 1000), type = "response"),
        balance_2000 = predict(lgD, data.frame(balance = 2000), type = "response"),
        rounded_1000 = plogis(-10.6513 + 0.0055 * 1000),
        rounded_2000 = plogis(-10.6513 + 0.0055 * 2000),
        mean_fitted_logit = mean(fitted(lgD))), 5)
round(predict(gD2, data.frame(student = c("Yes", "No")), type = "response"), 4)

ISLR's Tables 4.1 and 4.2 are reproduced. A unit of `balance` adds $0.0055$ to the log odds of default (Theorem 7.11); its $z=24.95$ is the Wald ratio of Corollary 7.21. The predicted probability at a balance of $1000$ is $0.00575$; ISLR's $0.00576$ uses the rounded slope $0.0055$, and at $2000$ the two are $0.5858$ and $0.5863$, ISLR's $0.586$. The logit's fitted probabilities average $0.0333$, the default rate, by Corollary 7.16; and with the one indicator `student` the fitted probabilities $0.0431$ and $0.0292$ are the two groups' default rates, by the same corollary.

**Example 7.11** (Confounding; ISLR §4.3.4, Table 4.3, Figure 4.3). With `income` in thousands:

In [ ]:
Df$income1000 <- Df$income / 1000
gD3 <- glm(default ~ balance + income1000 + student, family = binomial, data = Df)
round(coef(summary(gD3)), 4)
round(predict(gD3, data.frame(balance = 1500, income1000 = 40, student = c("Yes", "No")),
              type = "response"), 4)
round(rbind(default_rate = tapply(Df$y, Df$student, mean),
            mean_balance = tapply(Df$balance, Df$student, mean)), 4)
bins <- cut(Df$balance, c(0, 500, 1000, 1500, 2000, 2700), include.lowest = TRUE)
round(tapply(Df$y, list(bins, Df$student), mean), 4)

ISLR's Table 4.3 is reproduced. The `student` coefficient is $+0.405$ alone and $-0.647$ given `balance` and `income`: students default more often overall, $4.31\%$ against $2.92\%$, but at the same balance no more often: in every balance band of the table their default rate is at or below the non-students', because students carry larger balances, $988$ against $772$ on average. This is the same mechanism as Unit 3's theorem **omitted variables**, whose bias formula is for least squares and does not carry over to the logit. ISLR's predictions $0.058$ and $0.105$ for a student and a non-student with balance $1500$ and income $40$ are $0.0579$ and $0.1050$ from the fit.

## 4 Application: Medical Expenditures

> **Reading.** FREES §11.4. SRM learning outcomes 2(c), 2(e).

**Example 7.12** (Positive expenditure by explanatory variable; FREES §11.4, Table 11.4). For each level of each explanatory variable, the percentage of the $n=2000$ persons at that level and the percentage of them with positive inpatient expenditure, rounded half up as in FREES:

In [ ]:
pct1 <- function(k, m) floor(1000 * k / m + 0.5) / 10
vars <- c("GENDER", "RACE", "REGION", "EDUC", "PHSTAT", "MNHPOOR", "ANYLIMIT", "INCOME", "insure")
t114 <- do.call(rbind, lapply(vars, function(v) { k <- table(H[[v]]); s <- tapply(H$POSEXP, H[[v]], sum)
  data.frame(variable = v, level = names(k), pct_data = pct1(as.numeric(k), nrow(H)),
             pct_positive = pct1(as.numeric(s), as.numeric(k))) }))
print(t114, row.names = FALSE)
c(mean_age = round(mean(H$AGE), 1), pct_positive_all = pct1(sum(H$POSEXP), nrow(H)))

The table reproduces FREES's Table 11.4 to the printed digit, the mean age $39.0$ and the $7.9\%$ positive overall included. The largest percentages positive are for poor physical health, $36.0\%$, poor or fair mental health, $16.8\%$, and an activity limitation, $14.6\%$; the insured have $9.2\%$ against $3.1\%$ for the uninsured.

**Example 7.13** (Logit and probit fits; FREES §11.4, Table 11.5). The full logit model, the reduced logit model without `AGE` and `MNHPOOR`, and the reduced probit model, with $z$ ratios; for the probit both from the expected information, as `glm` reports, and from the observed information of Theorem 7.14:

In [ ]:
f_full <- POSEXP ~ AGE + GENDER + RACE + REGION + EDUC + PHSTAT + MNHPOOR + ANYLIMIT + INCOME + insure
full <- glm(f_full, family = binomial, data = H); red <- glm(f_red, family = binomial, data = H)
pro <- glm(f_red, family = binomial(link = "probit"), data = H)
Xp <- model.matrix(pro); ep <- drop(Xp %*% coef(pro)); yp <- H$POSEXP
l1 <- dnorm(ep) / pnorm(ep); l0 <- dnorm(ep) / pnorm(-ep)
z_obs <- coef(pro) / sqrt(diag(solve(crossprod(Xp, (yp * l1 * (l1 + ep) + (1 - yp) * l0 * (l0 - ep)) * Xp))))
nm <- names(coef(full)); pad <- function(v) setNames(v[nm], nm)
round(cbind(full = coef(full), z_full = coef(summary(full))[, 3], reduced = pad(coef(red)),
            z_red = pad(coef(summary(red))[, 3]), probit = pad(coef(pro)),
            z_expected = pad(coef(summary(pro))[, 3]), z_observed = pad(z_obs)), 3)
print(rbind(log_likelihood = round(sapply(list(full = full, reduced = red, probit = pro), logLik), 2),
            AIC = round(sapply(list(full = full, reduced = red, probit = pro), AIC), 2)), digits = 7)
LR <- 2 * as.numeric(logLik(full) - logLik(red))
round(c(LRT = LR, p_value = pchisq(LR, 2, lower.tail = FALSE), LRT_rounded_loglik = 2 * (-488.69 + 488.78),
        FREES_p_for_0.36 = pchisq(0.36, 2, lower.tail = FALSE)), 3)

The three fits reproduce FREES's Table 11.5: the logit coefficients and $z$ ratios to the printed digit and the log-likelihoods $-488.69$, $-488.78$ and $-486.98$. The AIC values $1021.39$, $1017.55$ and $1013.97$ differ in the last digit from FREES's $1021.38$, $1017.56$ and $1013.96$, which are $-2$ times the rounded log-likelihoods plus $2d$. FREES's probit $t$ ratios are those of the observed information ($4.178$ for `GENDER`, $4.805$ for `PHSTATPOOR`), not of the expected information that `glm` uses ($4.193$, $4.740$). The likelihood ratio statistic for `AGE` and `MNHPOOR` is $0.166$, with $p$-value $0.920$ on $2$ degrees of freedom; from the rounded log-likelihoods it is $0.180$, and FREES's $0.36$, with $p$-value $0.835$, is twice that. The conclusion stands: the two variables add nothing.

**Example 7.14** (Reading the fits; FREES §11.2.4, §11.3.2, §11.4). The adjusted odds ratio for sex against the raw one of Example 7.5, the ratios of the logit to the probit coefficients that are at least twice their standard errors, the overall likelihood ratio test, and the measures of fit of Definition 7.24:

In [ ]:
L_red <- as.numeric(logLik(red)); L0_meps <- as.numeric(logLik(glm(POSEXP ~ 1, family = binomial, data = H)))
big <- names(which(abs(coef(summary(red))[, 3]) > 2))
round(c(odds_ratio_sex_full = exp(coef(full)[["GENDER"]]), exp_0.733 = exp(0.733),
        odds_ratio_sex_raw = (p_w / (1 - p_w)) / (p_m / (1 - p_m))), 3)
round(coef(red)[big] / coef(pro)[big], 3)
R2cs <- 1 - exp(2 * (L0_meps - L_red) / nrow(H)); R2max <- 1 - exp(2 * L0_meps / nrow(H))
round(c(L0 = L0_meps, LRT_overall = 2 * (L_red - L0_meps), df = length(coef(red)) - 1,
        max_scaled_R2 = R2cs / R2max, pseudo_R2 = (L_red - L0_meps) / (0 - L0_meps)), 4)
signif(c(p_value_overall = pchisq(2 * (L_red - L0_meps), length(coef(red)) - 1, lower.tail = FALSE)), 3)

Adjusting for the other variables, women's odds of hospitalisation are $2.082$ times men's, against the raw $2.462$; FREES's $2.081$ is $e^{0.733}$, from the rounded coefficient. The ratios of the logit to the probit coefficients with $|z|>2$ lie between $1.751$ and $2.149$, above the slope factor $1.5958$ of Proposition 7.12(i), and six of the eight exceed the standard-deviation factor $1.8138$ of (ii). The overall likelihood ratio statistic is $122.8076$ on $19$ degrees of freedom, with a negligible $p$-value, and the max-scaled $R^2$ and the pseudo-$R^2$ are $0.1407$ and $0.1116$.

## 5 Nominal Dependent Variables

> **Reading.** FREES §11.5; ISLR §4.3.5 (PA). SRM learning outcomes 2(a), 2(c).

### 5.1 The generalised logit

**Definition 7.26** (Multicategory response; FREES §11.5). A **nominal** response $y_i$ takes one of the unordered values $1,\ldots,c$, $c\geqslant2$, with $\pi_{ij}=\operatorname{\mathsf{P}}(y_i=j)$ and $\sum_j\pi_{ij}=1$. With $y_{ij}=\mathbb{1}\{y_i=j\}$ the probability of the $i$th response is $\prod_j\pi_{ij}^{y_{ij}}$, and the log-likelihood of independent responses is $L=\sum_{i=1}^n\sum_{j=1}^cy_{ij}\ln\pi_{ij}$.

**Definition 7.27** (Generalised logit; FREES §11.5.1, ISLR (4.13) (softmax coding)). The **generalised logit** model has $V_{ij}=\mathbf{x}_i'\boldsymbol{\beta}_j$ with a parameter vector $\boldsymbol{\beta}_j$ for each category and

$$
\pi_{ij}=\frac{\exp(V_{ij})}{\sum_{h=1}^c\exp(V_{ih})} ,\qquad j=1,\ldots,c .
$$

**Proposition 7.28** (Normalisation; FREES §11.5.1, ISLR (4.10)–(4.12), (4.14)). Adding the same vector $\mathbf{a}$ to every $\boldsymbol{\beta}_j$ leaves every $\pi_{ij}$ unchanged, so the normalisation $\boldsymbol{\beta}_c=\mathbf{0}$ loses nothing. Under it, $\ln(\pi_{ij}/\pi_{ic})=\mathbf{x}_i'\boldsymbol{\beta}_j$, the log odds of category $j$ against the baseline $c$; if $\operatorname{rank}(\mathbf{X})=k+1$ the probabilities determine $\boldsymbol{\beta}_1,\ldots,\boldsymbol{\beta}_{c-1}$; and for $c=2$, $\pi_{i1}=\Lambda(\mathbf{x}_i'\boldsymbol{\beta}_1)$, the logit model.

**Proof.** Multiplying numerator and denominator by $\exp(\mathbf{x}_i'\mathbf{a})$ changes every $\boldsymbol{\beta}_h$ into $\boldsymbol{\beta}_h+\mathbf{a}$, and $\mathbf{a}=-\boldsymbol{\beta}_c$ gives $\boldsymbol{\beta}_c=\mathbf{0}$. Then $\pi_{ij}/\pi_{ic}=\exp(\mathbf{x}_i'\boldsymbol{\beta}_j)/\exp(0)$. If two normalised parameter sets give the same probabilities, $\mathbf{x}_i'\boldsymbol{\beta}_j=\mathbf{x}_i'\boldsymbol{\beta}_j^\dagger$ for every $i$, so $\mathbf{X}(\boldsymbol{\beta}_j-\boldsymbol{\beta}_j^\dagger)=\mathbf{0}$ and $\boldsymbol{\beta}_j=\boldsymbol{\beta}_j^\dagger$. For $c=2$, $\pi_{i1}=e^{\mathbf{x}_i'\boldsymbol{\beta}_1}/(e^{\mathbf{x}_i'\boldsymbol{\beta}_1}+1)=\Lambda(\mathbf{x}_i'\boldsymbol{\beta}_1)$. ∎

**Corollary 7.29** (The baseline does not matter; ISLR §4.3.5). With $\operatorname{rank}(\mathbf{X})=k+1$, the maximised log-likelihood and the fitted probabilities are the same whichever category is the baseline, and the estimates with baseline $h$ are $\mathbf b_j-\mathbf b_h$, $j=1,\ldots,c$, where $\mathbf b_1,\ldots,\mathbf b_c$ are those with baseline $c$ ($\mathbf b_c=\mathbf{0}$).

**Proof.** The map $(\boldsymbol{\beta}_1,\ldots,\boldsymbol{\beta}_c)\mapsto(\boldsymbol{\beta}_1-\boldsymbol{\beta}_h,\ldots,\boldsymbol{\beta}_c-\boldsymbol{\beta}_h)$ takes the parameter sets with $\boldsymbol{\beta}_c=\mathbf{0}$ one-to-one onto those with $\boldsymbol{\beta}_h=\mathbf{0}$, its inverse subtracting the $c$th vector, and by Proposition 7.28 (with $\mathbf{a}=-\boldsymbol{\beta}_h$) it leaves every $\pi_{ij}$ unchanged. $L$ depends on the parameters only through the $\pi_{ij}$ (Definition 7.26), so the map carries maximisers to maximisers, with the same probabilities and the same maximum. ∎

**Proposition 7.30** (Pairs of categories; FREES §11.5.1). In the generalised logit model, for categories $j\neq h$,

$$
\frac{\pi_{ij}}{\pi_{ih}}=\exp\{\mathbf{x}_i'(\boldsymbol{\beta}_j-\boldsymbol{\beta}_h)\} ,\qquad
\operatorname{\mathsf{P}}(y_i=j\mid y_i\in\{j,h\})=\Lambda\bigl(\mathbf{x}_i'(\boldsymbol{\beta}_j-\boldsymbol{\beta}_h)\bigr) ,
$$

and given $y_i\neq1$ the probabilities of categories $2,\ldots,c$ are a generalised logit model in those categories.

**Proof.** The common denominator cancels in $\pi_{ij}/\pi_{ih}$. Then $\operatorname{\mathsf{P}}(y_i=j\mid y_i\in\{j,h\})=\pi_{ij}/(\pi_{ij}+\pi_{ih})=e^u/(1+e^u)$ with $u=\mathbf{x}_i'(\boldsymbol{\beta}_j-\boldsymbol{\beta}_h)$. Given $y_i\neq1$, the probability of $j\geqslant2$ is $\pi_{ij}/(1-\pi_{i1})=\exp(V_{ij})/\sum_{h\geqslant2}\exp(V_{ih})$. ∎

**Proposition 7.31** (The intercept-only model; FREES §11.5.1). With $\mathbf{x}_i=1$, $\boldsymbol{\beta}_j=\alpha_j$ and $\alpha_c=0$, $\pi_j=e^{\alpha_j}/(1+\sum_{h<c}e^{\alpha_h})$ and $\alpha_j=\ln(\pi_j/\pi_c)$. If each category occurs $n_j>0$ times, the maximum likelihood estimate is $\hat\pi_j=n_j/n$, that is $\hat\alpha_j=\ln(n_j/n_c)$.

**Proof.** The first formula is Definition 7.27 with $e^{\alpha_c}=1$, and the second is Proposition 7.28. The map $\boldsymbol{\alpha}\mapsto(\pi_1,\ldots,\pi_c)$ is a bijection of $\mathbb{R}^{c-1}$ onto the probability vectors with positive entries, its inverse being $\alpha_j=\ln(\pi_j/\pi_c)$, and $L=\sum_jn_j\ln\pi_j$. The function $\ln x-x+1$ has derivative $1/x-1$, positive on $(0,1)$ and negative on $(1,\infty)$, so by the prerequisites' mean value theorem $\ln x\leqslant x-1$ with equality only at $x=1$. For such a vector, then,

$$
\sum_jn_j\ln\pi_j-\sum_jn_j\ln\frac{n_j}n=\sum_jn_j\ln\frac{\pi_j}{n_j/n}
\leqslant\sum_jn_j\Bigl(\frac{n\pi_j}{n_j}-1\Bigr)=n\sum_j\pi_j-n=0 ,
$$

with equality iff every $\pi_j=n_j/n$. ∎

**Theorem 7.32** (Score and concavity of the generalised logit). Under $\boldsymbol{\beta}_c=\mathbf{0}$ and with $\boldsymbol{\beta}=(\boldsymbol{\beta}_1',\ldots,\boldsymbol{\beta}_{c-1}')'$,

$$
\frac{\partial L}{\partial\boldsymbol{\beta}_j}=\sum_{i=1}^n\mathbf{x}_i(y_{ij}-\pi_{ij}) ,\qquad j<c ,
$$

and $L$ is concave, strictly when $\operatorname{rank}(\mathbf{X})=k+1$. At a maximum likelihood estimate with an intercept, the fitted counts $\sum_i\hat\pi_{ij}$ equal the observed counts $n_j$.

**Proof.** With $\mathbf{v}_i=(\mathbf{x}_i'\boldsymbol{\beta}_1,\ldots,\mathbf{x}_i'\boldsymbol{\beta}_{c-1})'$ and $A(\mathbf{v})=\ln(1+\sum_{h<c}e^{v_h})$, $\ln\pi_{ij}=v_{ij}-A(\mathbf{v}_i)$ for $j<c$ and $\ln\pi_{ic}=-A(\mathbf{v}_i)$; as $\sum_jy_{ij}=1$, $L=\sum_i\{\sum_{j<c}y_{ij}\mathbf{x}_i'\boldsymbol{\beta}_j-A(\mathbf{v}_i)\}$. Now $\partial A/\partial v_j=\pi_{ij}$, and the chain rule with $\partial\mathbf{v}_i/\partial\boldsymbol{\beta}_j$ acting through $\mathbf{x}_i$ gives the score. Differentiating again, $\partial^2A/\partial v_j\partial v_h=\pi_{ij}(\delta_{jh}-\pi_{ih})$, so for $\mathbf{u}\in\mathbb{R}^{c-1}$, $\mathbf{u}'\nabla^2A\,\mathbf{u}=\sum_{j<c}\pi_{ij}u_j^2-(\sum_{j<c}\pi_{ij}u_j)^2=\operatorname{var}(U)$, where $U$ takes the value $u_j$ with probability $\pi_{ij}$, $j<c$, and $0$ with probability $\pi_{ic}>0$; $\operatorname{var}(U)=0$ only if $U$ is constant, that is $\mathbf{u}=\mathbf{0}$. So $A$ is strictly convex, and along a direction $(\mathbf{d}_1',\ldots,\mathbf{d}_{c-1}')'$ the second derivative of $L$ is $-\sum_i\mathbf{u}_i'\nabla^2A(\mathbf{v}_i)\mathbf{u}_i\leqslant0$ with $\mathbf{u}_i=(\mathbf{x}_i'\mathbf{d}_1,\ldots,\mathbf{x}_i'\mathbf{d}_{c-1})'$, zero only if every $\mathbf{X}\mathbf{d}_j=\mathbf{0}$, which for $\operatorname{rank}(\mathbf{X})=k+1$ means $\mathbf{d}=\mathbf{0}$; the prerequisites' lemma **convexity from the derivative** gives the concavity and the strict concavity. At a maximiser the score vanishes; its intercept coordinates say $\sum_i\hat\pi_{ij}=n_j$ for $j<c$, and the counts for $j=c$ agree because both sides sum to $n$ over $j$. ∎

**Example 7.15** (Job turnover; FREES §11.5.1, Table 11.6). Valletta's generalised logit with baseline “no change in employment” has, for dismissal, tenure coefficient $-0.221$ and tenure $\times$ time trend coefficient $0.008$. In 1992, time $16$:

In [ ]:
round(c(tenure_coefficient_1992 = -0.221 + 16 * 0.008, factor = exp(-0.221 + 16 * 0.008),
        decline = 1 - exp(-0.221 + 16 * 0.008)), 3)

By Proposition 7.30, an extra year of tenure in 1992 multiplies the odds of dismissal against no change by $0.911$, a decline of $0.089$. FREES reads this as the dismissal probability, which moves by the same factor only if the probability of no change stays fixed.

**Example 7.16** (Marital status in the MEPS data). A generalised logit of marital status on age and sex, with `MARRIED` as the baseline; the odds against being married for ten more years of age:

In [ ]:
H$MARISTAT <- relevel(factor(H$MARISTAT), ref = "MARRIED")
gl <- nnet::multinom(MARISTAT ~ AGE + GENDER, data = H, trace = FALSE, reltol = 1e-12)
round(coef(gl), 4)
round(exp(10 * coef(gl)[, "AGE"]), 3)
rbind(observed = table(H$MARISTAT), fitted = round(colSums(fitted(gl)), 2))

Against being married, ten more years of age multiply the odds of never having married by $0.336$, those of being widowed by $3.153$, and those of being divorced or separated by $1.136$. Women have larger log odds of being divorced or separated ($0.5331$) and widowed ($1.2061$), and smaller of never having married ($-0.2572$). The fitted category counts equal the observed ones, as Theorem 7.32 states.

### 5.2 The multinomial logit

**Definition 7.33** (Multinomial logit; FREES §11.5.2). The **multinomial** (or **conditional**) **logit** model has explanatory variables $\mathbf{x}_{ij}$ that depend on the alternative and one parameter vector $\boldsymbol{\beta}$: $V_{ij}=\mathbf{x}_{ij}'\boldsymbol{\beta}$ and $\pi_{ij}=\exp(V_{ij})/\sum_h\exp(V_{ih})$, with log-likelihood $L=\sum_i\{\sum_jy_{ij}\mathbf{x}_{ij}'\boldsymbol{\beta}-\ln\sum_h\exp(\mathbf{x}_{ih}'\boldsymbol{\beta})\}$. The **mixed logit** has $V_{ij}=\mathbf{x}_{i1j}'\boldsymbol{\beta}+\mathbf{x}_{i2}'\boldsymbol{\beta}_j$.

**Proposition 7.34** (Generalised logit as a multinomial logit; FREES §11.5.2). Stacking $\boldsymbol{\beta}=(\boldsymbol{\beta}_1',\ldots,\boldsymbol{\beta}_c')'$ and letting $\mathbf{x}_{ij}$ be $\mathbf{x}_i$ in the $j$th block of $c$ blocks and $\mathbf{0}$ elsewhere gives $\mathbf{x}_{ij}'\boldsymbol{\beta}=\mathbf{x}_i'\boldsymbol{\beta}_j$: the generalised logit is the multinomial logit with these explanatory variables, and the mixed logit likewise.

**Proof.** $\mathbf{x}_{ij}'\boldsymbol{\beta}=\sum_h\mathbf{0}'\boldsymbol{\beta}_h+\mathbf{x}_i'\boldsymbol{\beta}_j$. For the mixed logit, append the blocks for $\mathbf{x}_{i2}$ to the vector $\mathbf{x}_{i1j}$. ∎

**Proposition 7.35** (Independence of irrelevant alternatives; FREES §11.5.2). In the multinomial logit model, $\ln(\pi_{ih}/\pi_{ik})=(\mathbf{x}_{ih}-\mathbf{x}_{ik})'\boldsymbol{\beta}$, which involves no other alternative.

**Proof.** The common denominator cancels in $\pi_{ih}/\pi_{ik}=\exp(\mathbf{x}_{ih}'\boldsymbol{\beta})/\exp(\mathbf{x}_{ik}'\boldsymbol{\beta})$. ∎

**Remark** (FREES §11.5.2). So adding or removing an alternative leaves the odds between any two others unchanged, a drawback when two alternatives are close substitutes.

**Example 7.17** (Choice of health insurance; FREES §11.5.2, Tables 11.7–11.9). Browne and Frees's CPS data: the odds of group insurance against none, overall and by disability, with and without the disability law; the odds ratios of Table 11.9 for the pairs not involving the baseline, computed as ratios of those against it (Proposition 7.30); and FREES's percentage readings:

In [ ]:
round(c(no_law = 59.3 / 20.1, law = 57.6 / 19.9, ratio = (57.6 / 19.9) / (59.3 / 20.1)), 3)
round(c(nondisabled_no = 64.2 / 20.5, nondisabled_law = 63.4 / 21.2, ratio_nd = (63.4 / 21.2) / (64.2 / 20.5),
        disabled_no = 24.5 / 17.6, disabled_law = 21.0 / 11.4, ratio_d = (21.0 / 11.4) / (24.5 / 17.6)), 3)
orb <- rbind(nondisabled = c(group = 0.825, nongroup = 1.053, government = 1.010),
             disabled = c(group = 1.419, nongroup = 0.953, government = 1.664))
round(cbind(group_vs_nongroup = orb[, 1] / orb[, 2], group_vs_government = orb[, 1] / orb[, 3],
            nongroup_vs_government = orb[, 2] / orb[, 3]), 3)
round(c(disabled_odds_change = 1.419 - 1, nondisabled_odds_change = 0.825 - 1, FREES_reciprocal = 1 / 0.825 - 1), 3)

From the rounded percentages the odds of group insurance against none are $2.950$ without the law and $2.894$ with it, a ratio of $0.981$, against FREES's $2.946$, $2.895$ and $0.983$. By disability the ratios are $0.955$ for the nondisabled and $1.323$ for the disabled (FREES $0.952$ and $1.329$): an effect in the disabled subgroup that the aggregate hides. The odds ratios of Table 11.9 that do not involve the uninsured follow from the others, $0.783$, $0.817$ and $1.043$ for the nondisabled and $1.489$, $0.853$ and $0.573$ for the disabled, against FREES's $0.784$, $0.818$, $1.043$, $1.490$, $0.854$ and $0.573$. The law multiplies the disabled's odds of group insurance against none by $1.419$, an increase of $0.419$, and the nondisabled's by $0.825$, a decrease of $0.175$; FREES's “$21.2\%$ less likely” is $1/0.825-1=0.212$, the increase of the odds when the law is not in effect.

### 5.3 The nested logit

**Definition 7.36** (Nested logit; FREES §11.5.3). For $\rho>0$ and $S_i=\sum_{h\geqslant2}\exp(V_{ih}/\rho)$, the **nested logit** has

$$
\pi_{i1}=\frac{\exp(V_{i1})}{\exp(V_{i1})+S_i^{\rho}} ,\qquad
\pi_{ij}=(1-\pi_{i1})\frac{\exp(V_{ij}/\rho)}{S_i} ,\quad j\geqslant2 .
$$

**Proposition 7.37** (The nested logit; FREES §11.5.3). (i) The $\pi_{ij}$ are positive and sum to $1$.

(ii) With $\rho=1$ they are the probabilities of Definition 7.27 with $V_{ij}$ in place of $\mathbf{x}_i'\boldsymbol{\beta}_j$.

(iii) For $j,h\geqslant2$, $\pi_{ij}/\pi_{ih}=\exp\{(V_{ij}-V_{ih})/\rho\}$, while $\pi_{i1}/\pi_{ij}=\exp(V_{i1}-V_{ij}/\rho)\,S_i^{1-\rho}$, which depends on every $V_{ih}$, $h\geqslant2$, $h\neq j$, unless $\rho=1$.

**Proof.** (i) $0<\pi_{i1}<1$, and $\sum_{j\geqslant2}\pi_{ij}=(1-\pi_{i1})S_i/S_i$. (ii) With $\rho=1$, $S_i=\sum_{h\geqslant2}e^{V_{ih}}$, so $\pi_{i1}=e^{V_{i1}}/\sum_he^{V_{ih}}$ and $1-\pi_{i1}=S_i/\sum_he^{V_{ih}}$, whence $\pi_{ij}=e^{V_{ij}}/\sum_he^{V_{ih}}$. (iii) The first ratio is immediate. As $1-\pi_{i1}=S_i^\rho/\{\exp(V_{i1})+S_i^\rho\}$, $\pi_{i1}/\pi_{ij}=\exp(V_{i1})S_i/\{S_i^\rho\exp(V_{ij}/\rho)\}$, and $S_i$ is strictly increasing in each $V_{ih}$, so $S_i^{1-\rho}$ is strictly increasing in each $V_{ih}$ when $\rho<1$ and strictly decreasing when $\rho>1$. ∎

## 6 Ordinal Dependent Variables

> **Reading.** FREES §11.6. SRM learning outcomes 2(a), 2(c).

### 6.1 The cumulative logit

**Definition 7.38** (Proportional odds model; FREES §11.6.1). An **ordinal** response takes the ordered values $1<\cdots<c$. With $\alpha_1<\cdots<\alpha_{c-1}$ and $\mathbf{x}_i$ now without the intercept, the **cumulative logit** or **proportional odds** model is

$$
\operatorname{logit}\operatorname{\mathsf{P}}(y_i\leqslant j)=\alpha_j+\mathbf{x}_i'\boldsymbol{\beta} ,\qquad j=1,\ldots,c-1 ,
$$

and the $\alpha_j$ are the **cut points**.

**Proposition 7.39** (Ordered cut points; FREES §11.6.1). With $\alpha_0=-\infty$ and $\alpha_c=+\infty$, the model gives $\pi_{ij}=\Lambda(\alpha_j+\mathbf{x}_i'\boldsymbol{\beta})-\Lambda(\alpha_{j-1}+\mathbf{x}_i'\boldsymbol{\beta})$, and these are positive for every $\mathbf{x}_i$ iff $\alpha_1<\cdots<\alpha_{c-1}$; with $\alpha_{j-1}=\alpha_j$ category $j$ has probability $0$.

**Proof.** $\pi_{ij}=\operatorname{\mathsf{P}}(y_i\leqslant j)-\operatorname{\mathsf{P}}(y_i\leqslant j-1)$, and applying $\Lambda$ to the model gives the difference. $\Lambda$ is strictly increasing by Proposition 7.8, so the difference is positive iff $\alpha_j>\alpha_{j-1}$, whatever $\mathbf{x}_i'\boldsymbol{\beta}$, which is added to both. ∎

**Theorem 7.40** (Latent variable and the cumulative probit; FREES §11.6.2). Let $y_i^*=-\mathbf{x}_i'\boldsymbol{\beta}+\varepsilon_i$ and $y_i=j$ when $\alpha_{j-1}<y_i^*\leqslant\alpha_j$. If $\varepsilon_i$ has distribution function $\Lambda$, the model of Definition 7.38 holds; if $\varepsilon_i$ is standard normal, $\operatorname{\mathsf{P}}(y_i\leqslant j)=\Phi(\alpha_j+\mathbf{x}_i'\boldsymbol{\beta})$, the **cumulative probit** model.

**Proof.** $\operatorname{\mathsf{P}}(y_i\leqslant j)=\operatorname{\mathsf{P}}(y_i^*\leqslant\alpha_j)=\operatorname{\mathsf{P}}(\varepsilon_i\leqslant\alpha_j+\mathbf{x}_i'\boldsymbol{\beta})$, which is $\Lambda(\alpha_j+\mathbf{x}_i'\boldsymbol{\beta})$ or $\Phi(\alpha_j+\mathbf{x}_i'\boldsymbol{\beta})$, and $\operatorname{logit}\Lambda(z)=z$. ∎

**Remark** (Signs; FREES §11.6.2). FREES takes $y_i^*-\mathbf{x}_i'\boldsymbol{\beta}$ logistic, which gives $\operatorname{logit}\operatorname{\mathsf{P}}(y_i\leqslant j)=\alpha_j-\mathbf{x}_i'\boldsymbol{\beta}$: its equation (11.12) with $-\boldsymbol{\beta}$ in place of $\boldsymbol{\beta}$. R's `MASS::polr` fits $\operatorname{logit}\operatorname{\mathsf{P}}(y_i\leqslant j)=\zeta_j-\mathbf{x}_i'\boldsymbol{\gamma}$, so $\zeta_j=\alpha_j$ and $\boldsymbol{\gamma}=-\boldsymbol{\beta}$. The conventions give the same probabilities and opposite signs: a positive $\gamma_l$ moves the response towards the higher categories.

**Theorem 7.41** (Proportional odds; FREES §11.6.1). In the model of Definition 7.38, raising $x_{il}$ by one unit, the other variables fixed, multiplies the cumulative odds $\operatorname{\mathsf{P}}(y_i\leqslant j)/\operatorname{\mathsf{P}}(y_i>j)$ by $e^{\beta_l}$ for every $j$, and $\beta_l=\partial\ln\operatorname{odds}(y_i\leqslant j)/\partial x_{il}$.

**Proof.** $\ln\operatorname{odds}(y_i\leqslant j)=\alpha_j+\mathbf{x}_i'\boldsymbol{\beta}$ increases by $\beta_l$, whatever $\alpha_j$, and its derivative in $x_{il}$ is $\beta_l$. ∎

**Corollary 7.42** (Dichotomising the response). For each $j<c$, $\mathbb{1}\{y_i\leqslant j\}$ follows the logit model with intercept $\alpha_j$ and slopes $\boldsymbol{\beta}$, the same slopes for every $j$.

**Proof.** $\operatorname{\mathsf{P}}(\mathbb{1}\{y_i\leqslant j\}=1)=\Lambda(\alpha_j+\mathbf{x}_i'\boldsymbol{\beta})$, which is Definition 7.6 with the logit and linear predictor $\alpha_j+\mathbf{x}_i'\boldsymbol{\beta}$. ∎

**Remark**. So the proportional odds restriction can be checked by fitting the $c-1$ dichotomised logit models: their slope vectors estimate the same $\boldsymbol{\beta}$.

**Example 7.18** (Health plan choice; FREES §11.6.1, Table 11.10). Pauly and Herring's cumulative logit of plan type, ordered from the most restrictive (HMO) to the least (fee for service), on $n=34{,}486$ persons, reports the odds ratio $1.064$ for women and the cut points as odds ratios $0.769$, $1.406$ and $12.089$:

In [ ]:
round(c(alpha = log(c(0.769, 1.406, 12.089)), female = log(1.064)), 3)

The cut points are $-0.263$, $0.341$ and $2.492$, increasing as Proposition 7.39 requires, and the coefficient for women is $\ln1.064=0.062$. FREES reads $1.064$ as women being more likely to choose less restrictive plans. Under equation (11.12) an odds ratio above $1$ multiplies the odds of the more restrictive categories $y\leqslant j$, so the reading holds in the `polr` convention of the remark on signs, where the reported ratio multiplies the odds of $y>j$.

**Example 7.19** (Self-rated physical health in the MEPS data). `PHSTAT` ordered from excellent to poor, on age, sex, activity limitation, insurance and income, by the cumulative logit and the cumulative probit, with $\boldsymbol{\beta}=-\boldsymbol{\gamma}$ from `polr`:

In [ ]:
H$PH <- factor(H$PHSTAT, levels = c("EXCE", "VGOO", "GOOD", "FAIR", "POOR"), ordered = TRUE)
f_ph <- PH ~ AGE + GENDER + ANYLIMIT + insure + INCOME
po <- MASS::polr(f_ph, data = H, Hess = TRUE); pp <- MASS::polr(f_ph, data = H, method = "probit", Hess = TRUE)
round(rbind(beta_logit = -coef(po), odds_ratio_higher = exp(coef(po)), beta_probit = -coef(pp),
            ratio = coef(po) / coef(pp)), 3)
round(rbind(alpha_logit = po$zeta, alpha_probit = pp$zeta), 3)
dich <- sapply(1:4, function(j) coef(glm(I(as.integer(PH) <= j) ~ AGE + GENDER + ANYLIMIT + insure + INCOME,
                                         family = binomial, data = H))[-1])
colnames(dich) <- paste0("y_le_", 1:4); round(cbind(dich, polr = -coef(po)), 3)

In the convention of Definition 7.38 (FREES's (11.12)) a negative coefficient moves the response towards poorer health: age ($-0.023$ per year), women ($-0.259$) and an activity limitation ($-1.197$, which multiplies the odds of a poorer category by $3.309$) do so, while insurance ($0.228$) and every income level above poor act the other way, the highest most ($1.339$). The cut points increase from $-0.898$ to $3.929$. The logit coefficients are $1.630$ to $1.775$ times the probit ones, except for the near-poor level, where both are near zero ($0.073$ and $0.017$). The dichotomised logits drift: the activity limitation coefficient goes from $-0.898$ at $y\leqslant1$ to $-2.110$ at $y\leqslant4$, and high income from $0.667$ to $2.311$, so by Corollary 7.42 the proportional odds restriction is doubtful for these variables.

## 7 Classification

> **Reading.** ISLR §4.3 and §4.3.3 (PA reading list): classification by a threshold on the fitted probability, $1/2$ or a lower one for a conservative lender. Beyond the syllabus: the confusion matrix, sensitivity, specificity, the ROC curve and the AUC (ISLR §4.4.2), and FREES §11, which fits probabilities and does not classify.

**Definition 7.43** (Classifier and confusion counts; ISLR §4.3, §4.4.2). With fitted probabilities $\hat p_i$ and a threshold $t\in(0,1)$, the classifier predicts $\hat y_i=\mathbb{1}\{\hat p_i>t\}$. The counts TP, FN, FP, TN of $(\hat y_i,y_i)$ equal to $(1,1),(0,1),(1,0),(0,0)$ (the confusion matrix, beyond the syllabus) give the **sensitivity** TP$/($TP$+$FN$)$, the **specificity** TN$/($TN$+$FP$)$ (these two beyond the syllabus) and the **accuracy** (TP$+$TN$)/n$.

**Theorem 7.44** (The cost-optimal threshold). Let $(\mathbf{X},y)$ have $\operatorname{\mathsf{P}}(y=1\mid\mathbf{X})=p(\mathbf{X})$, and let predicting $0$ when $y=1$ cost $c_{01}>0$ and predicting $1$ when $y=0$ cost $c_{10}>0$. Among classifiers $h(\mathbf{X})\in\{0,1\}$, the expected cost is minimised by $h^*(\mathbf{X})=\mathbb{1}\{p(\mathbf{X})>c_{10}/(c_{10}+c_{01})\}$.

**Proof.** By the defining property of the prerequisites' theorem **conditional expectation in $L^2$**, with $W$ the indicators $\mathbb{1}\{h(\mathbf{X})=0\}$ and $\mathbb{1}\{h(\mathbf{X})=1\}$, the expected cost is $\operatorname{\mathsf{E}}[c_{01}p(\mathbf{X})\mathbb{1}\{h(\mathbf{X})=0\}+c_{10}\{1-p(\mathbf{X})\}\mathbb{1}\{h(\mathbf{X})=1\}]$. For each value of $\mathbf{X}$ the integrand is smaller with $h=1$ iff $c_{10}\{1-p(\mathbf{X})\}<c_{01}p(\mathbf{X})$, that is $p(\mathbf{X})>c_{10}/(c_{10}+c_{01})$, and $h^*$ makes the integrand pointwise smallest. ∎

**Corollary 7.45**. With equal costs the threshold is $1/2$ and $h^*$ is Unit 1's Bayes classifier (definition **Bayes classifier**), which minimises the error rate (proposition **optimality of the Bayes classifier**).

**Proof.** $c_{10}/(c_{10}+c_{01})=1/2$, and with unit costs the expected cost is the error rate. ∎

**Definition 7.46** (ROC curve and AUC (beyond the syllabus)). For scores $s_i$ of $n_1$ observations with $y_i=1$ and $n_0$ with $y_i=0$, let $\mathrm{TPR}(t)=\#\{i:y_i=1,s_i\geqslant t\}/n_1$ and $\mathrm{FPR}(t)=\#\{i:y_i=0,s_i\geqslant t\}/n_0$. The **ROC curve** joins by segments the point $(0,0)$ and the points $(\mathrm{FPR}(t),\mathrm{TPR}(t))$ as $t$ decreases through the distinct scores, the last being $(1,1)$, and the **AUC** is the area under it.

**Theorem 7.47** (The area counts ordered pairs (beyond the syllabus)). $\mathrm{AUC}=\bigl[\#\{(i,j):y_i=1,y_j=0,s_i>s_j\}+\tfrac12\#\{(i,j):y_i=1,y_j=0,s_i=s_j\}\bigr]/(n_1n_0)$.

**Proof.** As $t$ decreases to a score value $v$ held by $a_v$ observations with $y=0$ and $b_v$ with $y=1$, the curve moves by $(a_v/n_0,b_v/n_1)$ from the previous point, of height $\#\{i:y_i=1,s_i>v\}/n_1$, and the trapezoid under this segment has area $(a_v/n_0)\{\#\{i:y_i=1,s_i>v\}+b_v/2\}/n_1$. Summing over $v$, the first terms count the pairs with $s_i>s_j$ and the second the ties, each tie counted with weight $1/2$. ∎

**Corollary 7.48**. A strictly increasing transformation of the scores leaves the ROC curve and the AUC unchanged, and so does any change of $n_1$ and $n_0$ that keeps the two within-class score distributions.

**Proof.** A strictly increasing map preserves every comparison $s_i>s_j$ and $s_i=s_j$ and the order of the thresholds. TPR and FPR are proportions within the two classes. ∎

**Example 7.20** (Classifying hospitalisation). The reduced logit fit of Example 7.13 as a classifier at $t=1/2$ and at the sample proportion of hospitalised persons, against the rule that predicts $0$ always; and the AUC (beyond the syllabus) of Theorem 7.47, of the trapezoids, and of the linear predictor:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
ph <- fitted(red); yv <- H$POSEXP
conf <- sapply(c(0.5, mean(yv)), function(t) { yh <- as.numeric(ph > t)
  c(t = t, TP = sum(yh == 1 & yv == 1), FN = sum(yh == 0 & yv == 1), FP = sum(yh == 1 & yv == 0),
    TN = sum(yh == 0 & yv == 0), sensitivity = mean(yh[yv == 1]), specificity = mean(1 - yh[yv == 0]),
    accuracy = mean(yh == yv)) })
colnames(conf) <- c("t_half", "t_rate"); round(conf, 4); round(c(accuracy_of_always_0 = mean(yv == 0)), 4)
pair_auc <- function(s) { d <- outer(s[yv == 1], s[yv == 0], "-"); mean((d > 0) + 0.5 * (d == 0)) }
thr <- sort(unique(ph), decreasing = TRUE)
tpr <- c(0, sapply(thr, function(t) mean(ph[yv == 1] >= t))); fpr <- c(0, sapply(thr, function(t) mean(ph[yv == 0] >= t)))
round(c(pairs = pair_auc(ph), trapezoids = sum(diff(fpr) * (head(tpr, -1) + tail(tpr, -1)) / 2),
        linear_predictor = pair_auc(qlogis(ph))), 4)
op <- par(mar = c(3.2, 3.8, 0.6, 0.6), mgp = c(2.1, 0.6, 0))
plot(fpr, tpr, type = "l", xlab = "false positive rate", ylab = "true positive rate"); abline(0, 1, lty = 3)
par(op)

*Figure 7.4.* ROC curve of the reduced logit model for hospitalisation, with the diagonal of a score unrelated to the response (dotted).

At $t=1/2$ the classifier finds $5$ hospitalised persons and misses $152$, a sensitivity of $0.0318$, and its accuracy $0.9200$ is below the $0.9215$ of predicting $0$ always. At the sample proportion $0.0785$ the sensitivity is $0.6943$ and the specificity $0.6934$. The AUC is $0.7588$ by pairs and by trapezoids, as Theorem 7.47 states, and for the linear predictor, as Corollary 7.48 states.

## 8 Computation in R

> **Reading.** FREES §11.4 and its chapter script.

### 8.1 FREES's chapter script

FREES's script tabulates the percentages of Example 7.12 with `Hmisc`, fits the logit on sex with `Design::lrm`, whose third statistic is the model likelihood ratio $2\{L(\mathbf{b})-L_0\}$, and compares the reduced and full logit models with `anova`; its full model omits `MNHPOOR`, and here the full model of Example 7.13 is used:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
c(model_LR_sex = g1$null.deviance - g1$deviance, df = 1)
anova(red, full, test = "Chisq")

The model likelihood ratio for sex alone is $26.38$ on $1$ degree of freedom, and `anova` repeats the statistic $0.166$ of Example 7.13, with $p$-value $0.92$.

### 8.2 Formulas against package output

The score at the fit, the standard errors from $\mathbf{X}'\mathbf{W}\mathbf{X}$, Fisher scoring (Theorem 7.18) for both links, the log-likelihood and AIC, and the Wald, $z^2$ and likelihood ratio statistics for sex (Corollary 7.21):

In [ ]:
X <- model.matrix(red); b <- coef(red); pr <- drop(plogis(X %*% b))
Info <- crossprod(X, pr * (1 - pr) * X)
scoring <- function(X, y, Fn, fn) { bb <- numeric(ncol(X))
  repeat { eta <- drop(X %*% bb); p <- Fn(eta); f <- fn(eta); w <- f^2 / (p * (1 - p)); z <- eta + (y - p) / f
    bn <- drop(solve(crossprod(X, w * X), crossprod(X, w * z))); if (max(abs(bn - bb)) < 1e-10) return(bn); bb <- bn } }
signif(c(max_abs_score = max(abs(crossprod(X, yv - pr))),
  se_manual_vs_glm = max(abs(sqrt(diag(solve(Info))) - coef(summary(red))[, 2])),
  scoring_logit_vs_glm = max(abs(scoring(X, yv, plogis, dlogis) - b)),
  scoring_probit_vs_glm = max(abs(scoring(X, yv, pnorm, dnorm) - coef(pro))),
  loglik_manual_minus_glm = sum(yv * log(pr) + (1 - yv) * log(1 - pr)) - as.numeric(logLik(red)),
  AIC_manual_minus_glm = -2 * as.numeric(logLik(red)) + 2 * ncol(X) - AIC(red)), 3)
j <- which(names(b) == "GENDER"); r0 <- glm(update(f_red, . ~ . - GENDER), family = binomial, data = H)
round(c(Wald = b[[j]]^2 / solve(Info)[j, j], z_squared = coef(summary(red))[j, 3]^2,
        LRT = 2 * as.numeric(logLik(red) - logLik(r0))), 4)

The score vanishes at the fit to $1.03\times10^{-9}$, the standard errors from $\mathbf{X}'\mathbf{W}\mathbf{X}$ agree with `glm`'s to $1.49\times10^{-6}$, Fisher scoring reproduces the logit fit to $8.07\times10^{-11}$ and the probit fit to $1.18\times10^{-5}$, and the log-likelihood and AIC agree exactly. For sex, $W=z^2=14.49$, and the likelihood ratio statistic, $15.42$, differs from it in this finite sample. Newton–Raphson for the generalised logit of Example 7.16, with $-\nabla^2L$, whose $(a,h)$ block $\sum_i\pi_{ia}(\delta_{ah}-\pi_{ih})\mathbf{x}_i\mathbf{x}_i'$ follows from $\partial^2A/\partial v_a\partial v_h$ in the proof of Theorem 7.32, against `nnet::multinom`; the pair `NEVMAR`–`MARRIED` fitted alone (Proposition 7.30); and the cumulative logit likelihood of Example 7.19 against `polr`:

In [ ]:
Xm <- model.matrix(~ AGE + GENDER, data = H); Ym <- model.matrix(~ MARISTAT - 1, data = H)
gl_newton <- function(X, Y) { q <- ncol(X); m <- ncol(Y) - 1; B <- matrix(0, q, m)
  repeat { V <- cbind(0, X %*% B); P <- exp(V) / rowSums(exp(V)); g <- as.vector(crossprod(X, Y[, -1] - P[, -1]))
    I <- matrix(0, q * m, q * m)
    for (a in 1:m) for (h in 1:m) I[(a - 1) * q + 1:q, (h - 1) * q + 1:q] <- crossprod(X, P[, a + 1] * ((a == h) - P[, h + 1]) * X)
    s <- solve(I, g); B <- B + matrix(s, q, m); if (max(abs(s)) < 1e-12) return(t(B)) } }
Bn <- gl_newton(Xm, Ym); V <- cbind(0, Xm %*% t(Bn)); Pm <- exp(V) / rowSums(exp(V))
signif(c(newton_vs_multinom = max(abs(Bn - coef(gl))), probabilities_vs_multinom = max(abs(Pm - fitted(gl))),
  loglik_newton_minus_multinom = sum(Ym * log(Pm)) - as.numeric(logLik(gl))), 3)
sub <- droplevels(subset(H, MARISTAT %in% c("MARRIED", "NEVMAR")))
round(rbind(generalised_logit = Bn[which(levels(H$MARISTAT) == "NEVMAR") - 1, ],
            two_category_logit = coef(glm(I(MARISTAT == "NEVMAR") ~ AGE + GENDER, family = binomial, data = sub))), 3)
Xo <- model.matrix(f_ph, data = H)[, -1]; cum <- sapply(po$zeta, function(a) plogis(a - drop(Xo %*% coef(po))))
Po <- cbind(cum, 1) - cbind(0, cum)
signif(c(loglik_manual_minus_polr = sum(log(Po[cbind(seq_len(nrow(Po)), as.integer(H$PH))])) - as.numeric(logLik(po)),
  probabilities_vs_predict = max(abs(Po - predict(po, type = "probs")))), 3)

Newton–Raphson agrees with `multinom` to $4.58\times10^{-7}$ in the coefficients and $4.28\times10^{-8}$ in the probabilities. The pair fitted alone gives $3.173$, $-0.105$ and $-0.250$ against $3.326$, $-0.109$ and $-0.257$ from the generalised logit, two estimates of the same vector under the model. The cumulative logit likelihood and probabilities agree with `polr` exactly.

### 8.3 The ISLR lab (beyond the syllabus)

> **Reading.** Beyond the syllabus: ISLR §4.7.1–4.7.2, the chapter's lab, which neither reading list includes.

**Example 7.21** (Stock market directions; ISLR §4.7.1–4.7.2). `Smarket`: $1250$ daily S&P 500 returns, 2001–2005, with the five previous returns `Lag1`–`Lag5`, `Volume` and `Direction` (`Up` or `Down`). The logit on all six, its classifications at threshold $1/2$, the fit on 2001–2004 tested on 2005, and the model with `Lag1` and `Lag2` only:

In [ ]:
Sm <- Smarket; CS <- cor(Sm[, -9]); round(CS["Year", "Volume"], 3)
diag(CS) <- 0; CS["Year", "Volume"] <- CS["Volume", "Year"] <- 0; round(max(abs(CS)), 3)
fS6 <- Direction ~ Lag1 + Lag2 + Lag3 + Lag4 + Lag5 + Volume
gS <- glm(fS6, data = Sm, family = binomial); round(coef(summary(gS)), 4)
cls <- function(p) ifelse(p > 0.5, "Up", "Down")
pS <- cls(fitted(gS)); table(pS, Sm$Direction); mean(pS == Sm$Direction)
trS <- Sm$Year < 2005; d05 <- Sm$Direction[!trS]
gS2 <- glm(fS6, data = Sm, family = binomial, subset = trS)
p2 <- cls(predict(gS2, Sm[!trS, ], type = "response")); table(p2, d05); mean(p2 != d05)
gS3 <- glm(Direction ~ Lag1 + Lag2, data = Sm, family = binomial, subset = trS)
p3 <- cls(predict(gS3, Sm[!trS, ], type = "response")); table(p3, d05)
round(c(accuracy = mean(p3 == d05), always_up = mean(d05 == "Up"),
        up_when_predicted_up = sum(p3 == "Up" & d05 == "Up") / sum(p3 == "Up")), 4)
round(predict(gS3, data.frame(Lag1 = c(1.2, 1.5), Lag2 = c(1.1, -0.8)), type = "response"), 4)

The only sizeable correlation among the variables is `Year` with `Volume`, $0.539$; every other is at most $0.048$ in absolute value. No coefficient is significant, the smallest $p$-value being `Lag1`'s $0.145$. The training accuracy is $52.2\%$; on 2005, after fitting to 2001–2004, the error rate is $52.0\%$. With `Lag1` and `Lag2` the 2005 accuracy is $56.0\%$, exactly that of predicting `Up` every day, and $58.2\%$ of the days predicted `Up` are `Up`. The predicted probabilities of `Up` at the two new days are $0.4791$ and $0.4961$. The training accuracy is the training error rate of Unit 1 subtracted from $1$, an optimistic estimate of the test accuracy, which the 2005 hold-out estimates.

## 9 Exercises

**Exercise 7.1** (FREES exercise 11.1). Let $y^*$ have the distribution function $F(y)=e^y/(1+e^y)$. (a) Find its density. (b) Find its mean $\mu_y$. (c) Find its standard deviation $\sigma_y$. (d) Find the density of $y^{**}=(y^*-\mu_y)/\sigma_y$. (e) Plot that density with the standard normal density.

**Solution.** (a) $F=\Lambda$, so the density is $F'(y)=e^y/(1+e^y)^2=\Lambda(y)\{1-\Lambda(y)\}$ by Proposition 7.8. (b) and (c) $\mu_y=0$ and $\sigma_y=\pi/\sqrt3$, with $\pi=3.14159\ldots$ the number, by Proposition 7.12(ii). (d) $\operatorname{\mathsf{P}}(y^{**}\leqslant a)=F(\sigma_ya)$, so $y^{**}$ has density $\sigma_yF'(\sigma_ya)=\sigma_ye^{\sigma_ya}/(1+e^{\sigma_ya})^2$. (e):

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.3)
s_y <- pi / sqrt(3); a <- seq(-4, 4, length = 801)
round(c(sigma_y = s_y, max_density_difference = max(abs(s_y * dlogis(s_y * a) - dnorm(a)))), 4)
op <- par(mar = c(3.2, 3.8, 0.6, 0.6), mgp = c(2.1, 0.6, 0))
plot(a, dnorm(a), type = "l", xlab = "y", ylab = "density", ylim = c(0, 0.47)); lines(a, s_y * dlogis(s_y * a), lty = 2)
par(op)

*Figure 7.5.* The density of $y^{**}$ (dashed) and the standard normal density (solid).

$\sigma_y=1.8138$, and the two densities differ by at most $0.0545$; the logistic density is the higher at the centre and in the tails. ■

**Exercise 7.2** (FREES exercise 11.2). Let $y_i^*=\mathbf{x}_i'\boldsymbol{\beta}+\varepsilon_i^*$ with $\varepsilon_i^*\sim N(0,\sigma^2)$ and $y_i=\mathbb{1}\{y_i^*>0\}$. Show that $\operatorname{\mathsf{P}}(y_i=1\mid\mathbf{x}_i)=\Phi(\mathbf{x}_i'\boldsymbol{\beta}/\sigma)$.

**Solution.** $\varepsilon_i^*=\sigma Z$ with $Z$ standard normal, so $\operatorname{\mathsf{P}}(y_i=1)=\operatorname{\mathsf{P}}(Z>-\mathbf{x}_i'\boldsymbol{\beta}/\sigma)=1-\Phi(-\mathbf{x}_i'\boldsymbol{\beta}/\sigma)=\Phi(\mathbf{x}_i'\boldsymbol{\beta}/\sigma)$, the normal density being even; this is Theorem 7.9 with $\pi=\Phi$. ■

**Exercise 7.3** (FREES exercise 11.3). Under the random utility interpretation with $V_{i2}=0$, $V_{i1}=\mathbf{x}_i'\boldsymbol{\beta}$ and errors with $\operatorname{\mathsf{P}}(\varepsilon_{ij}<a)=\exp(-e^{-a})$, show that $\pi_i=\operatorname{\mathsf{P}}(\varepsilon_{i2}-\varepsilon_{i1}<V_{i1}-V_{i2})=1/\{1+\exp(-\mathbf{x}_i'\boldsymbol{\beta})\}$.

**Solution.** The exercise leaves the joint law of the two errors open; with $\varepsilon_{i1}$ and $\varepsilon_{i2}$ independent, Proposition 7.10 gives $\operatorname{\mathsf{P}}(\varepsilon_{i2}-\varepsilon_{i1}\leqslant t)=\Lambda(t)$ for every $t$, and $\Lambda$ is continuous, so $\pi_i=\Lambda(\mathbf{x}_i'\boldsymbol{\beta})=1/\{1+\exp(-\mathbf{x}_i'\boldsymbol{\beta})\}$. ■

**Exercise 7.4** (FREES exercise 11.4). (a) For an i.i.d. Bernoulli sample with mean $\pi$, show that the maximum likelihood estimator of $\pi$ is $\bar y$. (b) For independent Bernoulli samples of sizes $n_1$ and $n_2$ with means $\pi_1$ and $\pi_2$: (i) show that the maximum likelihood estimator of $\pi_2-\pi_1$ is $\bar y_2-\bar y_1$; (ii) find its variance. (c) In the logit model with an intercept and a $0/1$ variable $x$, $n_1$ observations having $x=0$ and mean $\bar y_1$, and $n_2$ having $x=1$ and mean $\bar y_2$: (i) show that $\bar y_1=\pi(b_0)$ and $\bar y_2=\pi(b_0+b_1)$; (ii) show that $b_1=\pi^{-1}(\bar y_2)-\pi^{-1}(\bar y_1)$; (iii) confirm that $\mathcal{I}(\beta_0,\beta_1)=n_1\pi_1(1-\pi_1)\bigl(\begin{smallmatrix}1&0\\0&0\end{smallmatrix}\bigr) +n_2\pi_2(1-\pi_2)\bigl(\begin{smallmatrix}1&1\\1&1\end{smallmatrix}\bigr)$ with $\pi_1=\pi(\beta_0)$ and $\pi_2=\pi(\beta_0+\beta_1)$; (iv) find the large-sample variance of $b_1$.

**Solution.** (a) $L(\pi)=n\bar y\ln\pi+n(1-\bar y)\ln(1-\pi)$. If $0<\bar y<1$, $L'(\pi)=n(\bar y-\pi)/\{\pi(1-\pi)\}$ is positive below $\bar y$ and negative above, so $\bar y$ is the maximiser by the prerequisites' mean value theorem; if $\bar y=0$ or $1$, $L$ is monotone on $[0,1]$ with maximum at $\pi=\bar y$.

(b)(i) The log-likelihood is $L_1(\pi_1)+L_2(\pi_2)$, maximised at $(\bar y_1,\bar y_2)$ by (a), and $(\pi_1,\pi_2)\mapsto(\pi_1,\pi_2-\pi_1)$ is a bijection, so the maximiser in the new parameters is $(\bar y_1,\bar y_2-\bar y_1)$. (ii) By independence and Proposition 7.2, $\operatorname{var}(\bar y_2-\bar y_1)=\pi_1(1-\pi_1)/n_1+\pi_2(1-\pi_2)/n_2$.

(c)(i) and (ii) are Corollary 7.16, for $0<\bar y_1,\bar y_2<1$. (iii) By Theorem 7.14, $\mathcal{I}=\sum_i\pi_i(1-\pi_i)\mathbf{x}_i\mathbf{x}_i'$, with $\mathbf{x}_i\mathbf{x}_i'$ equal to $\bigl(\begin{smallmatrix}1&0\\0&0\end{smallmatrix}\bigr)$ for the $n_1$ observations with $x_i=0$ and to $\bigl(\begin{smallmatrix}1&1\\1&1\end{smallmatrix}\bigr)$ for the $n_2$ with $x_i=1$. (iv) With $u=n_1\pi_1(1-\pi_1)$ and $v=n_2\pi_2(1-\pi_2)$, $\mathcal{I}=\bigl(\begin{smallmatrix}u+v&v\\v&v\end{smallmatrix}\bigr)$ has the inverse $(uv)^{-1}\bigl(\begin{smallmatrix}v&-v\\-v&u+v\end{smallmatrix}\bigr)$, as multiplication verifies, with lower right entry $(u+v)/(uv)=1/u+1/v$; by Theorem 7.19 the large-sample variance of $b_1$ is $1/\{n_1\pi_1(1-\pi_1)\}+1/\{n_2\pi_2(1-\pi_2)\}$. For the fit of Example 7.7, with $\bar y_1$ and $\bar y_2$ in place of $\pi_1$ and $\pi_2$:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
n_m <- sum(tab_sex[, "0"]); n_w <- sum(tab_sex[, "1"])
round(c(se_formula = sqrt(1 / (n_m * p_m * (1 - p_m)) + 1 / (n_w * p_w * (1 - p_w))), se_glm = coef(summary(g1))[2, 2]), 6)

The formula and `glm` give the same standard error, $0.1837$. ■

**Exercise 7.5** (FREES exercise 11.5). With $\hat y_i=\pi(\mathbf{x}_i'\mathbf{b})$ the logit fitted values of a model with an intercept, show that $\bar y=n^{-1}\sum_i\hat y_i$.

**Solution.** The score equation for the intercept, Corollary 7.16, is $\sum_i\hat y_i=\sum_iy_i$. For the reduced model of Example 7.13: ■

In [ ]:
c(mean_response = mean(H$POSEXP), mean_fitted = mean(fitted(red)))

**Exercise 7.6** (FREES exercise 11.6). Beginning with the score equations (11.4) of FREES, verify the logit form (11.5).

**Solution.** Equation (11.4) is the score of Theorem 7.14, whose summands carry the factor $\pi'(\eta_i)/\{\pi_i(1-\pi_i)\}$. For the logit $\pi'=\Lambda'=\Lambda(1-\Lambda)$ by Proposition 7.8, so the factor is $1$ and the score is $\sum_i\mathbf{x}_i(y_i-\pi_i)$, equation (11.5). ■

**Exercise 7.7** (FREES exercise 11.7). (a) From the logit score, show that $\mathcal{I}(\boldsymbol{\beta})=\sum_i\sigma_i^2\mathbf{x}_i\mathbf{x}_i'$ with $\sigma_i^2=\pi_i(1-\pi_i)$. (b) From the general score (11.4), find the information matrix.

**Solution.** (a) Differentiating $\sum_i\mathbf{x}_i(y_i-\pi_i)$, with $\partial\pi_i/\partial\boldsymbol{\beta}'=\pi_i(1-\pi_i)\mathbf{x}_i'$, gives $\nabla^2L=-\sum_i\pi_i(1-\pi_i)\mathbf{x}_i\mathbf{x}_i'$, which contains no response and is minus its own expectation. (b) With $r=\pi'/\{\pi(1-\pi)\}$ the score is $\sum_i\mathbf{x}_i(y_i-\pi_i)r(\eta_i)$, and

$$
\nabla^2L=\sum_i\mathbf{x}_i\mathbf{x}_i'\{-\pi'(\eta_i)r(\eta_i)+(y_i-\pi_i)r'(\eta_i)\} ;
$$

the second term has mean zero, so $\mathcal{I}(\boldsymbol{\beta})=-\operatorname{\mathsf{E}}\nabla^2L=\sum_i\pi'(\eta_i)^2\mathbf{x}_i\mathbf{x}_i'/\{\pi_i(1-\pi_i)\}$, which is Theorem 7.14. ■

**Exercise 7.8** (FREES exercise 11.8). The IRC's $n=1340$ bodily injury claims; `ATTORNEY` is $1$ if the claimant is represented by an attorney and $2$ if not. The original file was withdrawn; `AUTOBIsim`, its simulated substitute, is used. (a)(i) Histograms and summary statistics of `CLMAGE` and `LOSS`, and a variable `lnLOSS`; (ii) their means by `ATTORNEY`; (iii) percentages of `ATTORNEY` by `CLMSEX`, `MARITAL`, `SEATBELT` and `CLMINSUR`; (iv) missing values. (b)(i) A logit model on `CLMSEX`, interpreted through odds; (ii) a logit model on `CLMAGE`, `CLMSEX`, `MARITAL`, `SEATBELT`, `CLMINSUR`: which variables are significant? (iii) who uses attorneys more, men or women? (iv) add `LOSS` and `lnLOSS`, keep the more important, and test it; (v) drop `CLMAGE` and compare, including the number of observations; (vi) the probability of an attorney for a single male aged $32$ wearing a seat belt, the driver insured, with economic loss \$5,000, under the model of (iv). (c) Part (b)(v) by probit, interpreting only signs.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.2)
bi <- read.csv("data/frees/csv/AUTOBIsim.csv"); bi$lnLOSS <- log(bi$LOSS); bi$att <- 1 * (bi$ATTORNEY == 1)
round(sapply(bi[, c("CLMAGE", "LOSS", "lnLOSS")], function(v) c(summary(v), sd = sd(v))), 3)
round(sapply(bi[, c("CLMAGE", "LOSS", "lnLOSS")], function(v) tapply(v, bi$ATTORNEY, mean)), 3)
lapply(bi[, c("CLMSEX", "MARITAL", "SEATBELT", "CLMINSUR")],
       function(v) round(100 * prop.table(table(ATTORNEY = bi$ATTORNEY, level = v), 2), 1))
colSums(is.na(bi))
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.4, 1.6, 0.6), mgp = c(2, 0.6, 0))
for (v in c("CLMAGE", "LOSS", "lnLOSS")) hist(bi[[v]], breaks = 40, main = v, xlab = "", col = "grey85", border = "white")
par(op)

*Figure 7.6.* Histograms of `CLMAGE`, `LOSS` and `lnLOSS` in `AUTOBIsim`.

In the substitute file `CLMAGE` is not an age: its median is $3.350$ and its maximum $161.100$, and the questions about age are answered for this variable as it stands. `LOSS` is skewed (mean $5.682$, median $2.000$, maximum $341.720$) and `lnLOSS` nearly symmetric (mean $0.603$, median $0.693$), so `lnLOSS` is the useful scale. Claimants with an attorney have larger losses (mean `lnLOSS` $1.309$ against $-0.131$) and slightly smaller `CLMAGE` ($5.354$ against $6.266$). Attorneys are used by $59.4\%$ of men and $44.9\%$ of women, $73.5\%$ of the divorced or separated, and $74.1\%$ of claimants whose driver was uninsured against $48.9\%$ of the others, while seat belts make little difference ($50.9\%$ and $54.5\%$). The substitute has no missing values.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
for (v in c("CLMSEX", "MARITAL", "SEATBELT", "CLMINSUR")) bi[[v]] <- factor(bi[[v]])
m1 <- glm(att ~ CLMSEX, family = binomial, data = bi); round(coef(summary(m1)), 4)
c(odds_ratio_female = exp(coef(m1)[["CLMSEX2"]]))
m2 <- glm(att ~ CLMAGE + CLMSEX + MARITAL + SEATBELT + CLMINSUR, family = binomial, data = bi); round(coef(summary(m2)), 4)
c(odds_ratio_female_model_ii = exp(coef(m2)[["CLMSEX2"]]))
m4 <- glm(att ~ CLMAGE + CLMSEX + MARITAL + SEATBELT + CLMINSUR + LOSS + lnLOSS, family = binomial, data = bi)
round(coef(summary(m4))[c("LOSS", "lnLOSS"), ], 4)
m4b <- glm(att ~ CLMAGE + CLMSEX + MARITAL + SEATBELT + CLMINSUR + lnLOSS, family = binomial, data = bi)
round(coef(summary(m4b)), 4)
m5 <- update(m4b, . ~ . - CLMAGE); round(coef(summary(m5)), 4)
c(n_model_iv = nobs(m4b), n_model_v = nobs(m5))
new8 <- data.frame(CLMAGE = 32, CLMSEX = factor(1, levels = 1:2), MARITAL = factor(2, levels = 1:4),
                   SEATBELT = factor(1, levels = 1:2), CLMINSUR = factor(2, levels = 1:2), lnLOSS = log(5))
c(probability_attorney = unname(predict(m4b, new8, type = "response")), share_CLMAGE_below_32 = mean(bi$CLMAGE < 32))
m5p <- glm(formula(m5), family = binomial(link = "probit"), data = bi); round(cbind(logit = coef(m5), probit = coef(m5p)), 4)

(i) Sex is significant ($z=-5.245$): women's odds of using an attorney are $0.5553$ times men's. (ii) `CLMSEX`, `CLMINSUR` and the divorced or separated level of `MARITAL` are significant at $5\%$ ($p$-values below $0.0001$, below $0.0001$ and $0.0109$); `CLMAGE` ($0.1532$), `SEATBELT` ($0.8875$) and the other marital levels are not. (iii) Men: the women's coefficient is $-0.5641$, so by Theorem 7.11(i) women's odds of using an attorney are $0.5689$ times men's, the other variables held fixed. (iv) `lnLOSS` has $z=14.564$ against $-2.425$ for `LOSS` and is kept; in the refit its $z$ is $15.156$. (v) Without `CLMAGE` the estimates move little (`CLMSEX` $-0.7101$ against $-0.7035$, `lnLOSS` $0.7984$ in both), and both fits use all $1340$ claims, the substitute having no missing values; in the original file `CLMAGE` is missing for some claims, and dropping it enlarges the sample. (vi) $0.7412$, at `CLMAGE` $=32$ on the substitute's scale, a value above the fraction $0.9881$ of its `CLMAGE` values: a prediction in the far tail. (c) Every probit coefficient has the sign of the logit one: women, claimants not wearing a seat belt and claimants whose driver was insured are less likely to use an attorney, and larger losses make it more likely. ■

**Exercise 7.9** (FREES exercise 11.9). Hong Kong horse racing: $925$ races, one horse drawn at random from the ten in each; `FINISH` $=1$ if it won, and `WIN` is the crowd's probability of its winning, read from the tote board. (a) A colleague would take two horses from each race: how are their responses related, and which assumption does this violate? (b) The mean of `FINISH`, the summary statistics of `WIN`, and the relation between the mean and the standard deviation of `FINISH`. (c) `WIN` by `FINISH`. (d) A linear probability model of `FINISH` on `WIN`: (i) significance; (ii) fit; (iii) can fitted values leave $[0,1]$? (e) A logistic model: is `WIN` significant? (f) Fitted values of both models at `WIN` $=0,0.01,0.05,0.10,1$, and a plot of one against the other. (g) Plot $\widehat{\texttt{FINISH}}-\texttt{WIN}$ against `WIN` and discuss a betting strategy.

**Solution.** (a) At most one horse wins a race, so the two responses have $y_1y_2=0$ and $\operatorname{cov}(y_1,y_2)=\operatorname{\mathsf{E}}[y_1y_2]-\pi_1\pi_2=-\pi_1\pi_2<0$: they are dependent, against the independence of the responses (Definition 7.1).

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.3)
hk <- read.csv("data/frees/csv/HKHorse.csv"); yb <- mean(hk$FINISH); nh <- nrow(hk)
round(c(mean_FINISH = yb, sd_FINISH = sd(hk$FINISH), formula = sqrt(nh * yb * (1 - yb) / (nh - 1)),
        mean_WIN = mean(hk$WIN), sd_WIN = sd(hk$WIN), min_WIN = min(hk$WIN), max_WIN = max(hk$WIN)), 4)
round(sapply(split(hk$WIN, hk$FINISH), function(v) c(n = length(v), mean = mean(v), sd = sd(v))), 4)
lpm9 <- lm(FINISH ~ WIN, data = hk); lg9 <- glm(FINISH ~ WIN, family = binomial, data = hk)
round(rbind(lpm = coef(summary(lpm9))[2, ], logit = coef(summary(lg9))[2, ]), 4)
round(c(R2_lpm = summary(lpm9)$r.squared, intercept_lpm = coef(lpm9)[[1]], at_WIN_1 = sum(coef(lpm9))), 4)
wv <- c(0, 0.01, 0.05, 0.10, 1); fl <- predict(lpm9, data.frame(WIN = wv)); fg <- predict(lg9, data.frame(WIN = wv), type = "response")
round(rbind(WIN = wv, lpm = fl, logit = fg, logit_over_WIN = fg / wv), 4)
wg <- seq(0.0005, 0.9995, by = 0.0005); up <- plogis(coef(lg9)[[1]] + coef(lg9)[[2]] * wg) > 1.25 * wg
ru <- rle(up); last <- cumsum(ru$lengths); first <- last - ru$lengths + 1
reg <- cbind(from = wg[first[ru$values]], to = wg[last[ru$values]])
cbind(reg, t(apply(reg, 1, function(ab) { k <- hk$WIN >= ab[1] & hk$WIN <= ab[2]
  c(horses = sum(k), won = sum(hk$FINISH[k]), expected_by_crowd = round(sum(hk$WIN[k]), 2)) })))
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.8, 0.6, 0.6), mgp = c(2.1, 0.6, 0))
plot(fitted(lg9), fitted(lpm9), pch = 16, cex = 0.4, xlab = "logit fitted", ylab = "LPM fitted"); abline(0, 1, lty = 3)
plot(hk$WIN, fitted(lg9) - hk$WIN, pch = 16, cex = 0.4, xlab = "WIN", ylab = "fitted - WIN"); abline(h = 0, lty = 3)
par(op)

*Figure 7.7.* Left: fitted values of the linear probability model against those of the logit model, with the identity dotted. Right: the logit fitted probability minus the crowd's probability, against the crowd's probability.

(b) The mean of `FINISH` is $0.0951$, near one in ten. For $0/1$ data $\sum_i(y_i-\bar y)^2=\sum_iy_i-n\bar y^2=n\bar y(1-\bar y)$, so the standard deviation is $\{n\bar y(1-\bar y)/(n-1)\}^{1/2}=0.2936$, larger than that of `WIN`, $0.0819$, though the means ($0.0996$ for `WIN`) are close. (c) Winners had mean `WIN` $0.1612$ against $0.0931$: the crowd's probabilities carry information. (d) `WIN` has $t=7.647$, but $R^2$ is $0.0596$; the fitted line is $0.0080$ at `WIN` $=0$ and $0.8830$ at $1$, so on $[0,1]$ it stays in $[0,1]$. (e) In the logit model `WIN` has $z=6.667$. (f) The fits differ most at the ends: $0.0080$ against $0.0398$ at `WIN` $=0$ and $0.8830$ against $0.9886$ at `WIN` $=1$. (g) The winners share a fraction $1-T$ of the pool in proportion to their stakes, so a unit bet on a horse carrying the fraction $x$ of the bets returns $(1-T)/x$ if it wins, and its expected gain is positive iff the winning probability exceeds $x/(1-T)$, $1.25x$ for $T=0.2$. On a grid of `WIN` the fitted logit probability exceeds $1.25\,\texttt{WIN}$ on two intervals, from $0.0005$ to $0.0435$ and from $0.4480$ to $0.7360$ (the ratio is $4.2796$ at $0.01$, and $1.1447$, $0.8171$ and $0.9886$ at $0.05$, $0.10$ and $1$): the model suggests betting on extreme long shots and on strong favourites. The data do not bear this out: the $274$ horses in the first interval won $6$ races, against $6.75$ expected from the crowd's probabilities, and the $4$ in the second won $1$, against $2.04$, so the suggestion rests on the logit's functional form rather than on excess wins. ■

**Exercise 7.10** (FREES exercise 11.10). Term life insurance: the $n=500$ SCF families with positive income, `FACEPOS` $=1$ for the $275$ that bought term life insurance. (a) Means of the explanatory variables by `FACEPOS`. (b) A linear probability model on `LINCOME`, `EDUCATION`, `AGE`, `GENDER` and the factor `MARSTAT`: (i) define it; (ii) comment on its quality; (iii) its three drawbacks. (c) A logistic model on the same variables: (i) which variables are significant, and why; (ii) which measure summarises the fit? (d) A logistic model on `LINCOME`, `EDUCATION` and `MARSTAT1`, the indicator of `MARSTAT` $=1$: (i) compare the two by a likelihood ratio test; (ii) who is more likely to buy, married or not, in terms of odds; (iii) the probability of buying for a married man aged $54$ with $13$ years of education, wages of \$70,000 and a household of four.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
tl <- read.csv("data/frees/csv/TermLife.csv"); tl$FACEPOS <- 1 * (tl$FACE > 0)
tl$LINCOME <- log(tl$INCOME); tl$MARSTAT1 <- 1 * (tl$MARSTAT == 1)
round(sapply(split(tl[, c("LINCOME", "EDUCATION", "AGE", "GENDER", "NUMHH", "MARSTAT1")], tl$FACEPOS), colMeans), 3)
lpm10 <- lm(FACEPOS ~ LINCOME + EDUCATION + AGE + GENDER + factor(MARSTAT), data = tl); round(coef(summary(lpm10)), 4)
round(c(R2 = summary(lpm10)$r.squared, min_fitted = min(fitted(lpm10)), max_fitted = max(fitted(lpm10)),
        outside_01 = sum(fitted(lpm10) < 0 | fitted(lpm10) > 1)), 4)
lg10 <- glm(FACEPOS ~ LINCOME + EDUCATION + AGE + GENDER + factor(MARSTAT), family = binomial, data = tl)
round(coef(summary(lg10)), 4)
L10 <- as.numeric(logLik(lg10)); L10_0 <- as.numeric(logLik(glm(FACEPOS ~ 1, family = binomial, data = tl)))
round(c(LRT_overall = 2 * (L10 - L10_0), pseudo_R2 = (L10 - L10_0) / (0 - L10_0), AIC = AIC(lg10)), 4)
lg10r <- glm(FACEPOS ~ LINCOME + EDUCATION + MARSTAT1, family = binomial, data = tl); round(coef(summary(lg10r)), 4)
LR10 <- 2 * as.numeric(logLik(lg10) - logLik(lg10r))
round(c(LRT = LR10, df = 3, critical_5pct = qchisq(0.95, 3), p_value = pchisq(LR10, 3, lower.tail = FALSE),
        odds_ratio_married = exp(coef(lg10r)[["MARSTAT1"]]),
        probability = unname(predict(lg10r, data.frame(LINCOME = log(70000), EDUCATION = 13, MARSTAT1 = 1), type = "response"))), 4)

(a) Buyers have larger mean `LINCOME` ($11.149$ against $10.650$) and `EDUCATION` ($14.524$ against $13.498$), and a larger share married ($0.756$ against $0.556$). (b)(i) The model of Definition 7.3, $\operatorname{\mathsf{P}}(\texttt{FACEPOS}=1)=\mathbf{x}'\boldsymbol{\beta}$, fitted by least squares. (ii) $R^2=0.0690$; only `EDUCATION` and married have $|t|>2$; the fitted values stay within $[0.1058,0.8012]$. (iii) The three drawbacks of Proposition 7.4. (c)(i) By the Wald test of Corollary 7.21, `EDUCATION` ($z=2.3467$) and married ($2.3598$) are significant at $5\%$; `LINCOME` ($1.4539$), `AGE`, `GENDER` and living with a partner are not. (ii) The overall likelihood ratio statistic $35.2147$, or the pseudo-$R^2$ $0.0512$ of Proposition 7.25; the AIC is $666.9241$. (d)(i) $H_0$: the coefficients of `AGE`, `GENDER` and living with a partner are zero, the reduced model, against $H_a$: at least one of them is not, the full model; reject when $\text{LRT}$ exceeds $7.8147$, the $95\%$ point of $\chi^2_3$. $\text{LRT}=0.4010$ ($p=0.9400$), so $H_0$ is not rejected. (ii) Married persons: their odds of buying are $2.0558$ times those of the others. (iii) $0.5947$; age, sex and household size do not enter the reduced model. ■

**Exercise 7.11** (FREES exercise 11.11). Smith and Schumacher's logistic regression of graduating with an actuarial concentration for $n=185$ students: intercept $-12.094$ (standard error $2.575$), `GENDER` $0.256$ ($0.407$), `PLACE%` $4.336$ ($1.657$), `MSAT` $0.008$ ($0.004$), `VSAT` $-0.002$ ($0.003$), `RANK` $4.442$ ($1.836$). (a) Compute the $t$ ratios and state which variables are significant. (b) With `GENDER` $=1$, compute the success probability for (i) `PLACE%` $=0.80$, `MSAT` $=680$, `VSAT` $=570$, `RANK` $=0.90$, and for the changes (ii) `PLACE%` $=0.60$, (iii) `MSAT` $=620$, (iv) `VSAT` $=540$, (v) `RANK` $=0.70$.

**Solution.**

In [ ]:
est <- c(intercept = -12.094, GENDER = 0.256, PLACE = 4.336, MSAT = 0.008, VSAT = -0.002, RANK = 4.442)
se11 <- c(2.575, 0.407, 1.657, 0.004, 0.003, 1.836); round(c(est / se11, z_0.975 = qnorm(0.975)), 3)
cases <- rbind(i = c(1, 1, 0.80, 680, 570, 0.90), ii = c(1, 1, 0.60, 680, 570, 0.90), iii = c(1, 1, 0.80, 620, 570, 0.90),
               iv = c(1, 1, 0.80, 680, 540, 0.90), v = c(1, 1, 0.80, 680, 570, 0.70))
round(plogis(drop(cases %*% est)), 4)

(a) By Corollary 7.21 with the normal reference, PLACE% ($2.617$), MSAT ($2.000$) and RANK ($2.419$) exceed $1.96$ in size and are significant at $5\%$, MSAT barely; GENDER ($0.629$) and VSAT ($-0.667$) are not. (b) The probabilities are $0.4822$ in case (i), $0.2812$ with PLACE% $0.60$, $0.3655$ with MSAT $620$, $0.4972$ with VSAT $540$ and $0.2769$ with RANK $0.70$. ■

**Exercise 7.12** (FREES exercise 11.12). Case-control sampling: from $N$ records, each with $y_i=1$ is kept with probability $\tau_1$ and each with $y_i=0$ with probability $\tau_0$, the event $\{r_i=1\}$. (a) Find $\operatorname{\mathsf{P}}(y_i=1,r_i=1)$, $\operatorname{\mathsf{P}}(y_i=0,r_i=1)$ and $\operatorname{\mathsf{P}}(r_i=1)$ in terms of $\tau_0,\tau_1$ and $\pi_i$. (b) Find $\operatorname{\mathsf{P}}(y_i=1\mid r_i=1)$. (c) Rewrite it for $\pi_i=\Lambda(\mathbf{x}_i'\boldsymbol{\beta})$. (d) Show that the likelihood of the kept responses is the logistic likelihood with a changed intercept, and give the intercept.

**Solution.** The selection depends on the record only through $y_i$: $\operatorname{\mathsf{P}}(r_i=1\mid y_i=1)=\tau_1$ and $\operatorname{\mathsf{P}}(r_i=1\mid y_i=0)=\tau_0$. (a) $\operatorname{\mathsf{P}}(y_i=1,r_i=1)=\tau_1\pi_i$, $\operatorname{\mathsf{P}}(y_i=0,r_i=1)=\tau_0(1-\pi_i)$, and their sum $\operatorname{\mathsf{P}}(r_i=1)=\tau_1\pi_i+\tau_0(1-\pi_i)$. (b) $\operatorname{\mathsf{P}}(y_i=1\mid r_i=1)=\tau_1\pi_i/\{\tau_1\pi_i+\tau_0(1-\pi_i)\}$. (c) Multiplying numerator and denominator by $1+e^{\eta_i}$, $\eta_i=\mathbf{x}_i'\boldsymbol{\beta}$, gives $\tau_1e^{\eta_i}/(\tau_1e^{\eta_i}+\tau_0)=\Lambda\{\eta_i+\ln(\tau_1/\tau_0)\}$. (d) Given the selection, the kept responses are independent with these probabilities, so their likelihood is that of Definition 7.13 with the logit and $\mathbf{x}_i'\boldsymbol{\beta}$ replaced by $(\beta_0+\ln(\tau_1/\tau_0))+\sum_{j\geqslant1}x_{ij}\beta_j$: the slopes are unchanged and the intercept becomes $\beta_0+\ln(\tau_1/\tau_0)$. A simulation with $\tau_1=1$ and $\tau_0=0.1$:

In [ ]:
set.seed(1112); N <- 2000000; xx <- rnorm(N); yy <- rbinom(N, 1, plogis(-4 + xx))
keep <- runif(N) < ifelse(yy == 1, 1, 0.1)
round(c(intercept_shift = coef(glm(yy[keep] ~ xx[keep], family = binomial))[[1]] + 4, log_10 = log(10),
        slope = coef(glm(yy[keep] ~ xx[keep], family = binomial))[[2]]), 3)

The estimated intercept shift is $2.287$, against $\ln10=2.303$, and the slope $1.010$, against $1$. ■

**Exercise 7.13** (SOA sample question 19). For $Y=\beta_0+\beta_1X_1+\beta_2X_2+\beta_3X_1X_2+\varepsilon$ the maximised log-likelihoods are $-1126.91$ with the intercept only, $-1122.41$ with the intercept, $X_1$ and $X_2$, and $-1121.91$ with all four terms. $H_0\colon\beta_1=\beta_2=\beta_3=0$ is tested at the $5\%$ level by the likelihood ratio test. Which is true? (A) The statistic is $1$ and $H_0$ cannot be rejected. (B) It is $9$ and $H_0$ cannot be rejected. (C) It is $10$ and $H_0$ cannot be rejected. (D) It is $9$ and $H_0$ should be rejected. (E) It is $10$ and $H_0$ should be rejected.

**Solution.** $H_0$ restricts the full model to the intercept-only model, $q=3$, so the statistic is $2\{-1121.91-(-1126.91)\}$, referred to $\chi^2_3$ by Proposition 7.22:

In [ ]:
round(c(LRT = 2 * (-1121.91 + 1126.91), critical_5pct = qchisq(0.95, 3), stat_1 = 2 * (-1121.91 + 1122.41),
        stat_9 = 2 * (-1122.41 + 1126.91), p_value = pchisq(10, 3, lower.tail = FALSE)), 4)

The statistic is $10.0000$, above the $5\%$ critical value $7.8147$ of $\chi^2_3$ ($p$-value $0.0186$), so $H_0$ is rejected: (E). The $1$ of (A) compares the full model with the model without the interaction and tests $\beta_3=0$ only; the $9$ of (B) and (D) compares the model without the interaction with the intercept-only one and tests $\beta_1=\beta_2=0$ given $\beta_3=0$; (C) has the right statistic and the wrong decision. ■

**Exercise 7.14** (SOA sample question 42). Which statement is NOT true about the linear probability, logistic and probit models for binary responses? (A) The three major drawbacks of the linear probability model are poor fitted values, heteroscedasticity and meaningless residual analysis. (B) The logistic and probit models aim to circumvent the drawbacks of linear probability models. (C) The logit function is $\pi(z)=e^z/(1+e^z)$. (D) The probit function is $\pi(z)=\Phi(z)$. (E) The logit and probit functions are substantially different.

**Solution.** (A) is true: Proposition 7.4. (B) is true: their fitted probabilities lie in $(0,1)$, and the likelihood of Definition 7.13 uses the Bernoulli law of each response, with its variance $\pi_i(1-\pi_i)$, instead of a least squares fit with constant variance and continuous errors. (C) and (D) are the definitions (Definition 7.6, Proposition 7.8). (E) is false: after rescaling by $\pi/\sqrt3$ the two distribution functions differ by at most $0.0227$, and $\operatorname{logit}(p)/\Phi^{-1}(p)$ stays between $1.5958$ and $1.7145$ on $[0.1,0.9]$ (Example 7.6). The answer is (E). ■

**Exercise 7.15** (SOA sample question 67). A succeeds $8$ times out of $10$ and B $2$ times out of $10$. A logistic model with linear term $\beta_0+\beta_1X$, $X$ the indicator of A, estimates success probabilities $0.8$ for A and $0.2$ for B; under $\beta_1=0$ both are $0.5$. By the likelihood ratio test, what is the lowest of the significance levels (A) $0.100$, (B) $0.050$, (C) $0.025$, (D) $0.010$, (E) $0.005$ at which $\beta_1=0$ is rejected?

**Solution.** By Corollary 7.16 the fitted probabilities are the two proportions and, under $\beta_1=0$, the pooled $10/20$, as stated. The full model gives probability $0.8$ to $8+8=16$ outcomes and $0.2$ to $4$; the reduced model $0.5$ to all $20$. The statistic is referred to $\chi^2_1$ (Corollary 7.21):

In [ ]:
LR67 <- 2 * ((16 * log(0.8) + 4 * log(0.2)) - 20 * log(0.5))
round(c(LRT = LR67, qchisq(1 - c(`0.100` = 0.1, `0.050` = 0.05, `0.025` = 0.025, `0.010` = 0.01, `0.005` = 0.005), 1)), 3)

The statistic $7.710$ exceeds the critical values of $\chi^2_1$ at $0.100$, $0.050$, $0.025$ and $0.010$ ($2.706$, $3.841$, $5.024$, $6.635$) but not the $7.879$ at $0.005$, so the lowest available level at which $H_0$ is rejected is $0.010$: (D). ■

**Exercise 7.16** (SOA sample question 72). A logistic model of positive medical expenditure gives: Age $0.257$ (standard error $0.102$, $p$-value $0.0117$); Sex $-0.955$ ($0.542$, $0.0781$); Education $0.103$ ($0.069$, $0.1355$); Log(Income) $-0.516$ ($0.157$, $0.0010$). Which predictors are significant at $5\%$? (A) Log(Income) only; (B) Education only; (C) Age and Log(Income) only; (D) Sex and Education only; (E) all four.

**Solution.** Each $p$-value is $2\{1-\Phi(|z|)\}$ with $z=b_j/\operatorname{se}(b_j)$, the Wald test of Corollary 7.21 with $q=1$:

In [ ]:
b72 <- c(Age = 0.257, Sex = -0.955, Education = 0.103, LogIncome = -0.516); s72 <- c(0.102, 0.542, 0.069, 0.157)
round(rbind(z = b72 / s72, p_value = 2 * pnorm(-abs(b72 / s72))), 4)

The $z$ ratios $2.5196$, $-1.7620$, $1.4928$ and $-3.287$ give back the $p$-values $0.0117$, $0.0781$, $0.1355$ and $0.001$. Age and Log(Income) are below $0.05$ and Sex and Education are not, so (C) holds, and (A), (B), (D) and (E) each misclassify at least one predictor. ■

**Exercise 7.17** (ISLR §4.8, exercise 1). Show that the logistic function representation $p(X)=e^{\beta_0+\beta_1X}/(1+e^{\beta_0+\beta_1X})$ and the logit representation $p(X)/\{1-p(X)\}=e^{\beta_0+\beta_1X}$ are equivalent.

**Solution.** Write $z=\beta_0+\beta_1X$. If $p=e^z/(1+e^z)$ then $1-p=1/(1+e^z)$ and $p/(1-p)=e^z$. If $p/(1-p)=e^z$ with $p\in(0,1)$, then $p=e^z(1-p)$, so $p(1+e^z)=e^z$ and $p=e^z/(1+e^z)$. This is $\operatorname{logit}\Lambda(z)=z$ of Proposition 7.8. ■

**Exercise 7.18** (ISLR §4.8, exercise 3). With one predictor and $K$ classes, let $X$ be normal in class $k$ with mean $\mu_k$ and variance $\sigma_k^2$, the variances differing between classes, and let $\pi_k$ be the proportion of class $k$. Show that the Bayes classifier is not linear in $x$ but quadratic.

**Solution.** Write $\pi_k$ for the class proportions and $f_k$ for the density of $X$ in class $k$, with $f=\sum_l\pi_lf_l>0$. **Bayes' theorem** in this form, $\operatorname{\mathsf{P}}(Y=k\mid X=x)=\pi_kf_k(x)/f(x)$ (ISLR (4.15)), holds because for every bounded $g$, $\operatorname{\mathsf{E}}[g(X)I(Y=k)]=\pi_k\int gf_k=\operatorname{\mathsf{E}}[g(X)\,\pi_kf_k(X)/f(X)]$, so $\pi_kf_k(X)/f(X)$ satisfies the defining relation of the conditional expectation of $I(Y=k)$ given $X$ (the prerequisites' lemma **conditional expectation in $L^1$**); with $p_k$ the Borel function of Unit 1's convention, the uniqueness argument of that lemma needs the relation only for $W=I\{h(X)>p_k(X)\}$ and its mirror, $h=\pi_kf_k/f$, which are bounded functions of $X$. Here $f_k$ is the normal density with mean $\mu_k$ and variance $\sigma_k^2$, and the denominator $f(x)$ is the same for every $k$, so the Bayes classifier (Unit 1) assigns $x$ to the $k$ maximising $\pi_kf_k(x)$, or its logarithm, which apart from a constant common to all classes is

$$
\delta_k(x)=\ln\pi_k-\ln\sigma_k-\frac{(x-\mu_k)^2}{2\sigma_k^2}
=-\frac{x^2}{2\sigma_k^2}+\frac{\mu_k}{\sigma_k^2}x-\frac{\mu_k^2}{2\sigma_k^2}+\ln\pi_k-\ln\sigma_k .
$$

The coefficient of $x^2$ depends on $k$, so the boundary $\delta_k(x)=\delta_l(x)$ between two classes is the solution of a quadratic equation in $x$; only when $\sigma_k=\sigma_l$ do the $x^2$ terms cancel and the boundary become linear. ■

**Exercise 7.19** (ISLR §4.8, exercise 4). Let $X$ be uniform on $[0,1]^p$, and predict at a test point from the training observations within $10\%$ of the range of each feature closest to it. (a) For $p=1$, what fraction of the observations is used on average? (b) For $p=2$? (c) For $p=100$? (d) What does this say about KNN for large $p$? (e) For a hypercube centred at the test point containing on average $10\%$ of the observations, find the side length for $p=1$, $2$, $100$.

**Solution.** (a) With the window $[x_0-0.05,x_0+0.05]$ clipped to $[0,1]$, its length is $0.1$ for $x_0\in[0.05,0.95]$ and $0.05+x_0$ or $1.05-x_0$ near the ends, so the average fraction is $0.1-2\int_0^{0.05}(0.05-t)\,\mathrm{d} t=0.1-0.0025=0.0975$; with the window shifted to stay inside $[0,1]$, as ISLR's “closest $10\%$ of the range” reads, it is $0.1$ exactly. (b) The features are independent, so the fraction is the product: $0.0975^2=0.0095$, or $0.01$. (c) $0.0975^{100}=8\times10^{-102}$, or $10^{-100}$. (d) For large $p$ a neighbourhood that is local in every coordinate contains essentially no training points, so the $K$ nearest neighbours are far away, Unit 6's proposition **neighbourhoods are not local**. (e) Side $s$ with $s^p=0.1$: $s=0.1$, $0.316$ and $0.1^{1/100}=0.977$; at $p=100$ the cube that holds $10\%$ of the data spans almost the whole range of every feature. ■

**Exercise 7.20** (ISLR §4.8, exercise 6). A logistic regression of receiving an A on $X_1=$ hours studied and $X_2=$ undergraduate GPA gives $\hat\beta_0=-6$, $\hat\beta_1=0.05$, $\hat\beta_2=1$. (a) Estimate the probability of an A for a student who studies $40$ hours with GPA $3.5$. (b) How many hours does that student need for a $50\%$ chance?

**Solution.** (a) $\Lambda(-6+0.05\cdot40+3.5)=\Lambda(-0.5)=1/(1+e^{0.5})=0.3775$. (b) $\Lambda(z)=1/2$ iff $z=0$: $-6+0.05h+3.5=0$, $h=50$ hours. ■

**Exercise 7.21** (ISLR §4.8, exercise 7). Companies are to be classified by whether they issue a dividend ($Y=$ Yes or No) from last year's percentage profit $X$. $80\%$ issued dividends; $X$ is normal with mean $10$ for those that did and $0$ for those that did not, with variance $36$ in both. Predict the probability that a company with $X=4$ issues a dividend.

**Solution.** By Bayes' theorem in the form of Exercise 7.18's solution, with $\phi$ the normal density with standard deviation $6$,

$$
\operatorname{\mathsf{P}}(Y=\text{Yes}\mid X=4)=\frac{0.8\,\phi(4-10)}{0.8\,\phi(4-10)+0.2\,\phi(4-0)}
=\frac{0.8e^{-1/2}}{0.8e^{-1/2}+0.2e^{-2/9}}=0.752 .
$$

The class probability of Unit 1's Bayes classifier is computed here from the class densities and the prior proportions, the route of ISLR's §4.4, rather than modelled directly as in logistic regression. ■

**Exercise 7.22** (ISLR §4.8, exercise 8). A data set is split into equal training and test halves. Logistic regression has training error $20\%$ and test error $30\%$; 1-nearest neighbours has error $18\%$ averaged over the two halves. Which method should be preferred for new observations?

**Solution.** With distinct predictor values, each training point is its own nearest neighbour, so the 1-NN training error is $0$ and the average $18\%$ means a test error of $36\%$. Logistic regression's test error, $30\%$, is lower, and the test error estimates the error on new observations (Unit 1); prefer logistic regression. ■

**Exercise 7.23** (ISLR §4.8, exercise 9). (a) What fraction of people with odds $0.37$ of defaulting default on average? (b) What are the odds for a person with a $16\%$ chance of defaulting?

**Solution.** By Definition 7.7, odds $o$ correspond to $p=o/(1+o)$. (a) $0.37/1.37=0.270$. (b) $0.16/0.84=0.190$. ■

**Exercise 7.24** (ISLR §4.8, exercise 12). You fit $\widehat{\operatorname{\mathsf{P}}}(\texttt{orange}\mid x)=\Lambda(\hat\beta_0+\hat\beta_1x)$ and a friend fits the softmax coding with $\hat\alpha_{\texttt{orange}0},\hat\alpha_{\texttt{orange}1}, \hat\alpha_{\texttt{apple}0},\hat\alpha_{\texttt{apple}1}$. (a) What are the log odds of orange against apple in your model? (b) In your friend's? (c) If $\hat\beta_0=2$, $\hat\beta_1=-1$, what are your friend's estimates? (d) If your friend's are $1.2$, $-2$, $3$, $0.6$, what are yours? (e) On $2000$ test observations, how often do the two models' predicted labels agree?

**Solution.** (a) $\hat\beta_0+\hat\beta_1x$. (b) $(\hat\alpha_{\texttt{orange}0}-\hat\alpha_{\texttt{apple}0}) +(\hat\alpha_{\texttt{orange}1}-\hat\alpha_{\texttt{apple}1})x$ (Proposition 7.30). (c) Only the differences are determined: $\hat\alpha_{\texttt{orange}0}-\hat\alpha_{\texttt{apple}0}=2$ and $\hat\alpha_{\texttt{orange}1}-\hat\alpha_{\texttt{apple}1}=-1$; adding a common vector to both classes changes no probability (Proposition 7.28), so the four numbers are not identified. (d) $\hat\beta_0=1.2-3=-1.8$ and $\hat\beta_1=-2-0.6=-2.6$ (Corollary 7.29). (e) The two fits give the same probability at every $x$, so the labels agree on all $2000$ observations, except at a probability of exactly $1/2$. ■

**Exercise 7.25** (ISLR §4.8, exercise 13, parts (a)–(d), (g), (i), and (j) without LDA, QDA and naive Bayes). With `Weekly` ($1089$ weekly returns, 1990–2010): (a) summarise the data; (b) fit the logit of `Direction` on the five lags and `Volume`, and identify any significant predictor; (c) give the confusion matrix and the fraction of correct predictions; (d) fit on 1990–2008 with `Lag2` only and evaluate on 2009–2010; (g) repeat (d) with KNN, $K=1$; (i) which method is best? (j) Experiment with other predictors, transformations and interactions in the logit, and other values of $K$ for KNN, reporting the test accuracy. Parts (e), (f), (h) and the rest of (j) use LDA, QDA and naive Bayes, which are on neither reading list and are not covered.

**Solution.**

In [ ]:
suppressMessages(library(class)); Wk <- Weekly
round(c(n = nrow(Wk), cor_year_volume = cor(Wk$Year, Wk$Volume), up = mean(Wk$Direction == "Up")), 3)
gW <- glm(Direction ~ Lag1 + Lag2 + Lag3 + Lag4 + Lag5 + Volume, data = Wk, family = binomial)
round(coef(summary(gW)), 4)
pW <- cls(fitted(gW)); table(pW, Wk$Direction); mean(pW == Wk$Direction)
trW <- Wk$Year <= 2008; dW <- Wk$Direction[!trW]
gW2 <- glm(Direction ~ Lag2, data = Wk, family = binomial, subset = trW)
pW2 <- cls(predict(gW2, Wk[!trW, ], type = "response")); table(pW2, dW)
set.seed(1); kW <- knn(as.matrix(Wk$Lag2[trW]), as.matrix(Wk$Lag2[!trW]), Wk$Direction[trW], k = 1)
table(kW, dW)
round(c(logit = mean(pW2 == dW), knn1 = mean(kW == dW), always_up = mean(dW == "Up")), 4)

(a) `Volume` grows with `Year` (correlation $0.842$); $55.6\%$ of the weeks are `Up`. (b) Only `Lag2` is significant at $5\%$ ($z=2.18$, $p=0.030$). (c) $56.1\%$ of the weeks are classified correctly, but $987$ of the $1089$ are predicted `Up`: almost every `Down` week ($430$ of $484$) is misclassified, while $557$ of the $605$ `Up` weeks are right. (d) On 2009–2010, $62.5\%$ correct, against $58.7\%$ for always predicting `Up`. (g) KNN with $K=1$ is right $50.0\%$ of the time. (i) The logit on `Lag2` is best of the two, and only modestly better than predicting `Up` every week.

In [ ]:
trw <- Wk$Year <= 2008
accw <- function(f) mean(ifelse(predict(glm(f, binomial, Wk[trw, ]), Wk[!trw, ],
  type = "response") > 0.5, "Up", "Down") == Wk$Direction[!trw])
round(c(lag2 = accw(Direction ~ Lag2), lag1_lag2 = accw(Direction ~ Lag1 + Lag2),
        lag2_squared = accw(Direction ~ Lag2 + I(Lag2^2)),
        lag2_times_volume = accw(Direction ~ Lag2 * Volume)), 4)
set.seed(1)
round(sapply(c(K1 = 1, K5 = 5, K10 = 10, K20 = 20, K50 = 50), function(K)
  mean(knn(Wk[trw, "Lag2", drop = FALSE], Wk[!trw, "Lag2", drop = FALSE],
           Wk$Direction[trw], k = K) == Wk$Direction[!trw])), 4)

(j) Adding `Lag2`$^2$ leaves the logit at $62.5\%$, and adding `Lag1` or the interaction with `Volume` lowers it; KNN improves with $K$ up to about $20$ ($59.6\%$) but never reaches the logit on `Lag2`. On $104$ test weeks each accuracy has standard error about $\sqrt{0.6\cdot0.4/104}=0.048$ (Proposition 7.2, the weeks treated as independent), and a difference of two independent accuracies about $\sqrt2$ times that, $0.068$ (differences of models scored on the same weeks have smaller standard errors, as low as $0.014$ here, and the conclusion stands for them too); even the largest gap, $12.5$ points between the logit and KNN with $K=1$, is under two such standard errors, so none of the differences is clearly more than noise, and the simplest model does as well as any. ■

**Exercise 7.26** (ISLR §4.8, exercise 14, parts (a)–(c), (f), (h)). With `Auto`: (a) create `mpg01`, $1$ when `mpg` exceeds its median; (b) find the features most associated with it; (c) split into training and test halves; (f) fit a logistic regression on those features and give its test error; (h) do the same with KNN for several $K$. Parts (d), (e), (g) use LDA, QDA and naive Bayes and are not covered.

**Solution.**

In [ ]:
Au <- Auto; Au$mpg01 <- 1 * (Au$mpg > median(Au$mpg))
round(cor(Au[, c("mpg01", "cylinders", "displacement", "horsepower", "weight",
                 "acceleration", "year")])[1, -1], 3)
set.seed(1); teA <- sample(nrow(Au), nrow(Au) / 2)
gA <- glm(mpg01 ~ cylinders + displacement + horsepower + weight, data = Au[-teA, ],
          family = binomial)
XA <- scale(Au[, c("cylinders", "displacement", "horsepower", "weight")])
knn_err <- sapply(c(1, 3, 5, 10, 20, 50), function(K) { set.seed(1)
  mean(knn(XA[-teA, ], XA[teA, ], Au$mpg01[-teA], k = K) != Au$mpg01[teA]) })
round(c(logit = mean((predict(gA, Au[teA, ], type = "response") > 0.5) != Au$mpg01[teA]),
        setNames(knn_err, paste0("knn", c(1, 3, 5, 10, 20, 50)))), 4)

(a), (b) `cylinders`, `displacement`, `weight` and `horsepower` have correlations $-0.76$ to $-0.67$ with `mpg01`, far larger than `acceleration` and `year`. (c) A random half of the $392$ cars is the test set. (f) The logit's test error is $9.7\%$. (h) On the standardised features KNN's test error is $6.1\%$ at $K=5$, the best of the values tried, and $8.7\%$ at $K=1$. ■

**Exercise 7.27** (ISLR §4.8, exercise 15). Write functions: (a) `Power()` printing $2^3$; (b) `Power2(x, a)` printing $x^a$; (c) use it for $10^3$, $8^{17}$, $131^3$; (d) `Power3(x, a)` returning $x^a$; (e) plot $f(x)=x^2$ for $x=1,\ldots,10$, with a logarithmic axis; (f) `PlotPower(x, a)` plotting $x$ against $x^a$.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
Power <- function() print(2^3); Power()
Power2 <- function(x, a) print(x^a); Power2(3, 8); Power2(10, 3); Power2(8, 17); Power2(131, 3)
Power3 <- function(x, a) { result <- x^a; return(result) }
PlotPower <- function(x, a, ...) plot(x, Power3(x, a), xlab = "x", ylab = paste0("x^", a), ...)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.8, 1.4, 0.6), mgp = c(2.1, 0.6, 0), cex = 0.8)
PlotPower(1:10, 2, log = "y", main = "f(x) = x^2, log y axis"); PlotPower(1:10, 3, main = "x^3")
par(op)

`Power2` prints; `Power3` returns a value that can be used in further computation, which is what `PlotPower` needs. $8^{17}=2.25\times10^{15}$ is printed in floating point. ■

**Exercise 7.28** (ISLR §4.8, exercise 16, logistic regression and KNN). With `Boston`, predict whether a tract's crime rate is above the median, by logistic regression and KNN on subsets of the predictors. (LDA and naive Bayes, also asked for, are not covered.)

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
Bo <- Boston; Bo$crim01 <- 1 * (Bo$crim > median(Bo$crim))
round(cor(Bo)[setdiff(names(Bo), c("crim", "crim01")), "crim01"], 2)
set.seed(1); teB <- sample(nrow(Bo), nrow(Bo) / 2)
err_logit <- function(f) { g <- glm(f, data = Bo[-teB, ], family = binomial)
  mean((predict(g, Bo[teB, ], type = "response") > 0.5) != Bo$crim01[teB]) }
XB <- scale(Bo[, c("nox", "rad", "dis", "age", "indus", "tax")])
knnB <- sapply(c(1, 3, 5, 10, 20), function(K) { set.seed(1)
  mean(knn(XB[-teB, ], XB[teB, ], Bo$crim01[-teB], k = K) != Bo$crim01[teB]) })
round(c(logit_all = err_logit(crim01 ~ . - crim),
        logit_six = err_logit(crim01 ~ nox + rad + dis + age + indus + tax),
        setNames(knnB, paste0("knn", c(1, 3, 5, 10, 20)))), 4)

The predictors most correlated with `crim01` are `nox`, `rad`, `age`, `tax` and `indus` (positively) and `dis` (negatively). On a random test half, the logit on all predictors errs on $10.7\%$ of the tracts and on the six correlated ones $13.8\%$; KNN on those six, standardised, does best at $K=3$ with $7.5\%$, and worsens as $K$ grows to $20$ ($12.3\%$). ■

**Exercise 7.29** (ISLR §5.4, exercise 5). On `Default`, (a) fit the logit of `default` on `income` and `balance`; (b) estimate its test error by the validation set approach; (c) repeat with three other splits; (d) add `student` and compare.

**Solution.** The validation set approach is Unit 1's; the classifier uses threshold $1/2$:

In [ ]:
vs <- t(sapply(1:4, function(sd) { set.seed(sd); tr <- sample(nrow(Df), nrow(Df) / 2)
  e <- function(f) { g <- glm(f, data = Df, family = binomial, subset = tr)
    mean((predict(g, Df[-tr, ], type = "response") > 0.5) != Df$y[-tr]) }
  c(seed = sd, income_balance = e(default ~ income + balance),
    with_student = e(default ~ income + balance + student)) }))
round(vs, 4)

The four splits give validation error rates between $2.38\%$ and $2.64\%$: the estimate varies with the split by about a quarter of a percentage point, Unit 1's first drawback of the validation set approach. Adding `student` raises the error slightly on every split ($2.46\%$ to $2.72\%$), so the dummy does not reduce the test error. ■

**Exercise 7.30** (ISLR §5.4, exercise 6; beyond the syllabus). For the logit of `default` on `income` and `balance`, compare the standard errors of `glm` with bootstrap standard errors.

**Solution.** The bootstrap is Unit 1's (beyond the syllabus):

In [ ]:
suppressMessages(library(boot))
gIB <- glm(default ~ income + balance, data = Df, family = binomial)
bfun <- function(d, i) coef(glm(default ~ income + balance, data = d[i, ], family = binomial))[2:3]
set.seed(1); bt <- boot(Df, bfun, R = 1000)
signif(rbind(glm_se = coef(summary(gIB))[2:3, 2], bootstrap_se = apply(bt$t, 2, sd)), 4)

The two agree to within $3\%$ ($4.99\times10^{-6}$ against $4.87\times10^{-6}$ for `income`, $2.27\times10^{-4}$ against $2.30\times10^{-4}$ for `balance`), as Theorem 7.19 leads one to expect when the logit is correctly specified: both estimate the sampling standard deviation of the estimate. ■

**Exercise 7.31** (ISLR §5.4, exercise 7). On `Weekly`: (a) fit the logit of `Direction` on `Lag1` and `Lag2`; (b) refit without the first observation; (c) classify the first observation; (d)–(e) compute the leave-one-out error with a loop.

**Solution.** LOOCV is Unit 1's, with the misclassification indicator as the loss (ISLR (5.4)):

In [ ]:
gL <- glm(Direction ~ Lag1 + Lag2, data = Wk, family = binomial); round(coef(gL), 4)
g_1 <- glm(Direction ~ Lag1 + Lag2, data = Wk[-1, ], family = binomial)
c(p_up_first = round(predict(g_1, Wk[1, ], type = "response"), 4),
  actual_first = as.character(Wk$Direction[1]))
loo <- sapply(seq_len(nrow(Wk)), function(i) {
  gi <- glm(Direction ~ Lag1 + Lag2, data = Wk[-i, ], family = binomial)
  cls(predict(gi, Wk[i, ], type = "response")) != Wk$Direction[i] })
round(c(loocv_error = mean(loo), always_up_error = mean(Wk$Direction != "Up")), 4)

(a) The coefficients are $0.2212$, $-0.0387$ and $0.0602$. (c) Without it, the first week gets probability $0.5714$ of `Up` and is classified `Up`; it was `Down`, an error. (e) The LOOCV error is $45.0\%$, against $44.4\%$ for always predicting `Up`: the lags do not help. ■